# Q27-AGENT v1.7 — Qwen3.8-27B REPL tool agent + DIRECT rescue (HYBRID), 4×L4

**Why.** Our direct-answer pipeline (Q27-XHIGH) scored LB 18.75: the model reasons up to 49k tokens and writes the
grid. The Kaggle discussions attribute the October jump to 60+ to Qwen3.8-27B *agents*; the public
`madarshbb/arc-qwen-3-8-27b-agent` notebook contains a REPL agent modelled on the Tufa Labs duck harness (Tufa Labs is
#1 at 88). This notebook runs that agent on our model.

**How the agent works** (code copied verbatim from the source notebook, cells 2–11):
* two tools: `python(code)` explores the task (grids preloaded as numpy arrays + helpers `show`, `diff`, `objects`);
  `submit(code)` runs a `transform(grid)` program on every train input and returns which examples it reproduced plus a
  cell diff. Reproducing all of them verifies the program; its test outputs become the answer.
* many short turns instead of one 49k-token monologue; per-turn thinking cap (`thinking_token_budget`), early-submit
  nudges, forced submit after 10 python calls, reasoning-effort ladder after a cut turn, function retention, context
  trimming with hysteresis (keeps the vLLM prefix cache valid), repetition-loop guard.
* answers: verified programs vote; otherwise the best unverified programs (most train examples right).
* serving: vLLM OpenAI server from our wheelhouse + the `taaf-duck-qwen38-serving-v1` transformers overlay,
  tool parser `qwen3_coder`, reasoning parser `qwen3`, prefix caching, fp8 KV, MTP (ladder falls back to plain).

**v1.7 (GPT final pre-GPU audit: NO-GO for v1.6; every finding checked in code first):**
* the EVAL score is now the DEPLOYED policy (DIRECT only where the agent did not verify -- what the hidden run does);
  the all-tasks-rescued merge is reported beside it (`hybrid_all_rescued_pct`), with agent-only, direct-only, the
  oracle union and how often the merge lost a correct agent / direct answer.
* `verified_but_wrong_tasks` / `_queries` come from the AGENT-only submission (a rescue no longer hides them).
* throughput per stage (agent, rescue, combined); DIRECT answers checkpointed to `rescue_attempts.jsonl` plus a
  deployed snapshot after every answer.
* guard: an install failure or an inactive hook (canary) now stops the child (v1.6 was fail-open); ctypes symbols for
  file / process functions, cffi and `_posixsubprocess.fork_exec` are blocked; sqlite3 / ctypes.dlopen paths checked.
  Still a best-effort guard for model-written code, not a security sandbox; denials are listed in the report.
* `train` (preloaded by the python tool) is now a tool-only name too; the check is scope-aware (symtable): a
  parameter, local or program-defined `train` is fine.
* agent end-to-end probe after the smoke: the agent's exact request (effort, budget, 8,192 max tokens, tools) does
  python -> tool result -> submit that verifies; if the configured effort fails, the first ladder effort that passes
  is used. A rejected rescue request shape disables the rescue (the agent keeps the time).
* circuit breaker also trips if the last 12 attempts all died on request errors (server lost mid-run).
* rerun: scout / deep caps 16k / 32k (EVAL keeps 24k / 36k and reports which verifications fit 16k); the rescue
  starts with tasks the agent never finished an attempt on.

**v1.6 (GPT deep audit DIRECT vs AGENT; each claim checked first):**
* HYBRID: the agent runs until `agent_frac` of the time (rerun 0.5, eval 0.6); then every task without a verified
  program gets a DIRECT answer (`rescue.py` → `run_direct.answer`, xhigh, 32,768 thinking + 4,096 answer tokens, one
  no-thinking commit call if cut). Merge per test input: verified grids → direct grid → unverified grids → test input.
  The commit run rescues EVERY task and reports agent-only / direct-only / hybrid official scores and their overlap.
* isolation: a PEP 578 audit hook in every child (sandbox + python tool) denies opening / listing any path containing
  `/kaggle/input` or `solution` and blocks subprocess / os.system / socket connect (the generated code cannot remove
  it; v1.7 made its failure stop the child); the regex scan of executed code stays as a second, report-only signal.
* two-tier budget `attempt_caps` [24k scout, 36k deep] instead of one 60k cap per attempt.
* circuit breaker: if the first 6 finished attempts all failed on HTTP errors with 0 tokens (run 1's failure mode),
  the agent stops and the time goes to the rescue.
* a submitted `transform` that uses the python-tool-only names `train_inputs` / `train_outputs` / `test_inputs` is
  rejected with feedback instead of failing silently in the sandbox.

**v1.5 (commit run 1 on esfersami50, 2026-10-11):** the infrastructure passed (libcuda probe PASS, rung 0 = MTP + fp8 KV
+ reasoning-config up in 490 s, every smoke stage incl. the roundtrip and the 6,144 budget OK in 64 s), but every
agent request failed: Qwen3.8's chat template rejects `reasoning_effort: "high"` (only xhigh / medium / low). Now
effort `xhigh` (template default; LB 18.75 and ARC Prize 42.4 % used it) with ladder [`medium`, `low`]; the smoke also sends the agent's EXACT request (its effort, sampling, budget,
tools; 32 output tokens) for its own effort and every ladder step, and a rejection stops the ladder (AGENT_PARAMS: a
config error, the same on every rung); the client no longer treats any 400 mentioning "reasoning" as a budget
rejection (run 1 dropped the budget because of that message).

**v1.4 (external re-audit; verified first):** the sandbox and the python tool no longer use `preexec_fn` (Python
documents a deadlock risk with threads; both run from 16 worker threads) -- the child sets its own RLIMIT_AS from
`Q27_CHILD_MAX_AS`; `pip install` has a timeout (wheelhouse 30 min, overlay 10 min); the GPU-drain wait after killing a
server is bounded (240 s, each nvidia-smi call 20 s).

**v1.3 (external audit R3; each point verified first):** the tool smoke adds a ROUNDTRIP on the mode the agent
will use (python call -> tool result sent back as role "tool" -> second request -> a `submit` that verifies, without
tool_choice), so a server that cannot take tool results is rejected; the thinking test sends the agent's real
`thinking_token_budget` (6,144; effort low to stay short); the python check uses a fresh random grid (a hard-coded
print cannot pass); the client keeps ONE absolute deadline per logical request including its budget-rejection retry,
and a runtime budget drop is printed and reported (`thinking_budget_dropped_at_runtime`); a rung that fails to start
is classified from the server log (CUDA_LINKER / UNSUPPORTED_FLAGS / OOM / UNKNOWN) and a CUDA-linker failure stops
the ladder (no rung changes the attention backend, so it would repeat); the file-access detector is wider (still
detection for review, not a sandbox).

**v1.2 (external audit R2; each point reproduced first):** the tool smoke now runs the returned code (a submit
must verify on a tiny identity task and return the right test output; a python call must run and print the shape), the
thinking test fails if the client had to drop `thinking_token_budget` (then the agent runs with thinking off and no
budget, explicitly), the whole smoke has one deadline (<= 180 s, each request <= 75 s) inside the agent reserve, a
failed `-lcuda` probe is reported loudly (not fatal: only FlashInfer JIT needs it); install/start/smoke times and the
time left for the agent are in `agent_server_report.json`; `progress.json` is written atomically; the eval report adds
`verified_but_wrong_tasks` and `file_access_suspicious_tasks` (generated code that touched files/paths, since the
evaluation solutions file exists on disk); the last cell is rewritten to repair bad ROWS (non-dict file, null/str rows,
ragged or out-of-range grids, unknown keys); the notebook is written with LF newlines so its SHA-256 is reproducible.

**v1.1 (external audit R1; every point reproduced or checked in code before the change):**
* server: libcuda shim + `-lcuda` link probe (Q27-XHIGH run 2 failed without it), `--max-num-batched-tokens 4096` and
  `VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=1` (run-2 OOM fix); each ladder rung must pass a real tool-calling smoke
  test (forced `submit`, automatic `python`, automatic with thinking) before the agent starts -- a failed forced test
  turns forced submits off, a failed thinking test runs the agent without thinking; the ladder stops when less than
  `min_agent_s` (eval 30 min, rerun 60 min) would remain; server PID file + SIGTERM handler, and the launcher kills a
  leftover vLLM process group after `submit_main` ends or is killed; `agent_server_report.json`.
* agent: every request is capped by the attempt's remaining tokens; attempt 2 of a task is queued only after attempt 1
  ended unverified (never both at once); a tool call without a usable `code` string gets targeted feedback; markup
  recovery keeps a final `<parameter>` closed only by `</function>`; non-dict tool arguments are normalised.
* budget: per-attempt cap = token supply / tasks (was x1.3); per turn 6,144 thinking / 8,192 total, 30 turns, nudge
  every 4 python calls, forced submit after 5 (were 8,192 / 12,288 / 40 / 6 / 10: seven full turns filled the 60k cap
  before a forced submit could happen).
* eval report: `official_score_pct` = correct test outputs / all test outputs (Kaggle's metric; the old task average
  is kept as `task_macro_score_pct`); fresh `attempts.jsonl` per run; `progress.json` after every attempt.
* last cell: grids must be 1..30 on each side with plain ints 0-9; the repair is written atomically.

**Modes** (same notebook):
* commit / interactive run: the agent on the 12 EVAL12 evaluation tasks (the same tasks as Q27-XHIGH run 7: direct =
  3/22 queries), budget 2.5 h from the first cell; report in `/kaggle/working/agent_eval/`.
* competition rerun: every hidden test task; `submission.json` is rewritten as each attempt finishes; the agent stops
  at 11 h 35 min and the process is cut at 11 h 50 min (10 min before the 12 h limit). A valid fallback submission is
  written in the first cell, and the last cell repairs any malformed row.


In [ ]:
# Runs first and cannot fail the notebook: a valid submission exists before anything heavy.
import json, glob, time, os
from pathlib import Path
START = time.time()
Path("/kaggle/working/start_epoch.txt").write_text(str(START))
try:
    ch = json.load(open(glob.glob("/kaggle/input/**/arc-agi_test_challenges.json", recursive=True)[0]))
    with open("/kaggle/working/submission.json", "w") as f:
        json.dump({t: [{"attempt_1": x["input"], "attempt_2": x["input"]} for x in task["test"]]
                   for t, task in ch.items()}, f)
    print("fallback submission written for", len(ch), "tasks; rerun =",
          os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
except Exception as e:
    print("fallback FAILED:", repr(e))
os.makedirs("/kaggle/working/agent", exist_ok=True)


In [ ]:
%%writefile /kaggle/working/agent/grids.py
"""Grid <-> text, task prompts and failure feedback for the program-synthesis agent.

Grids are shown as rows of digits with no separators, the same layout NVARC's
formatter uses, plus explicit dimensions -- reasoning models miscount rows and
columns of long digit strings otherwise.
"""

SYSTEM = (
    "You are an expert at solving ARC-AGI puzzles by writing Python programs. "
    "Each puzzle has a few input->output example pairs that share one hidden "
    "transformation rule. Grids are 2D lists of integers 0-9 (colours); 0 is "
    "usually the background."
)

INSTRUCTIONS = """Find the transformation rule that maps EVERY example input to its output, then write it as a Python function.

Requirements:
- Define `def transform(grid: list[list[int]]) -> list[list[int]]:`.
- Use only the Python standard library and numpy (already importable as `np` if you `import numpy as np`).
- The function must work for every example input AND the test input(s); do not hard-code example outputs.
- Return a new 2D list of ints 0-9 (numpy arrays are accepted).

Reason about the rule first, then give the final program in ONE ```python code block."""


def grid_text(g):
    return "\n".join("".join(str(int(v)) for v in row) for row in g)


def shape(g):
    return (len(g), len(g[0]) if g else 0)


def show(g, name):
    r, c = shape(g)
    return f"{name} ({r}x{c}):\n{grid_text(g)}"


def task_prompt(task):
    parts = [INSTRUCTIONS, ""]
    for i, pair in enumerate(task["train"], 1):
        parts.append(f"## Example {i}")
        parts.append(show(pair["input"], "Input"))
        parts.append(show(pair["output"], "Output"))
        parts.append("")
    for j, t in enumerate(task["test"], 1):
        parts.append(f"## Test input {j}")
        parts.append(show(t["input"], "Input"))
        parts.append("")
    return "\n".join(parts).rstrip()


def diff_text(expected, got, max_cells=900):
    """Expected grid with wrong cells marked, so the model sees WHERE it failed.

    '.' marks a correct cell; a wrong cell shows `expected/got` inline is too
    wide, so two aligned grids are given: the expected output, and the
    program's output with correct cells replaced by '.'.
    """
    re_, ce = shape(expected)
    rg, cg = shape(got)
    if (re_, ce) != (rg, cg):
        return (f"Wrong output SIZE: expected {re_}x{ce}, your program returned {rg}x{cg}.\n"
                + show(expected, "Expected") + "\n" + show(got, "Your output"))
    if re_ * ce > max_cells:
        wrong = sum(expected[r][c] != got[r][c] for r in range(re_) for c in range(ce))
        return f"{wrong} of {re_ * ce} cells are wrong (grid too large to print the diff)."
    wrong = 0
    rows = []
    for r in range(re_):
        s = []
        for c in range(ce):
            if expected[r][c] == got[r][c]:
                s.append(".")
            else:
                s.append(str(int(got[r][c])))
                wrong += 1
        rows.append("".join(s))
    return (f"{wrong} of {re_ * ce} cells are wrong.\n"
            + show(expected, "Expected") + "\n"
            + f"Your output, with CORRECT cells shown as '.' ({re_}x{ce}):\n" + "\n".join(rows))


def feedback_prompt(results, train, max_examples=3):
    """Tell the model which examples failed and how; ask for a corrected program."""
    lines = []
    n_ok = sum(r["ok"] for r in results)
    lines.append(f"Your program reproduced {n_ok} of {len(results)} examples.")
    shown = 0
    for i, (r, pair) in enumerate(zip(results, train), 1):
        if r["ok"]:
            continue
        if shown >= max_examples:
            break
        shown += 1
        lines.append(f"\n## Example {i} failed")
        if r.get("error"):
            lines.append(f"Error: {r['error']}")
        else:
            lines.append(diff_text(pair["output"], r["output"]))
    lines.append("\nRe-examine the rule against ALL examples, then give the corrected "
                 "complete program in ONE ```python code block.")
    return "\n".join(lines)


In [ ]:
%%writefile /kaggle/working/agent/sandbox.py
"""Run model-written `transform` programs in isolated subprocesses.

Each job is (code, inputs). The program runs in a fresh interpreter with a
wall-clock timeout (and, on Linux, an address-space cap), so an infinite loop,
a crash or a memory blow-up costs one job, never the run. Anything the program
prints is swallowed; the result travels back as one marked JSON line.

Per input the result is {"output": grid} or {"error": message}. An output is
only accepted if it is a 2D rectangular grid of ints 0-9, 1..30 on each side.
"""
import json
import os
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor

MARK = "@@ARC_RESULT@@"

RUNNER = r'''
try:                                    # v1.4: memory cap set by the child itself (no preexec_fn)
    import os as _q27_os, resource as _q27_res
    _q27_lim = int(_q27_os.environ.get("Q27_CHILD_MAX_AS", "0"))
    if _q27_lim > 0:
        _q27_res.setrlimit(_q27_res.RLIMIT_AS, (_q27_lim, _q27_lim))
except Exception:
    pass
# v1.6 guard: a PEP 578 audit hook (cannot be removed by the program). v1.7 (GPT final audit): installation failure or
# an inactive hook now ABORTS the child (it was fail-open), native routes are narrowed (ctypes symbols for file /
# process functions, cffi, _posixsubprocess.fork_exec), sqlite3/ctypes.dlopen paths are checked. This is a best-effort
# guard for model-written code, NOT a security sandbox (Python documents audit hooks as unsuitable for that).
try:
    import sys as _q27_sys, os as _q27_os2, re as _q27_re
    _q27_deny = tuple(p for p in _q27_os2.environ.get("Q27_DENY_PATHS", "/kaggle/input|solution").lower().split("|") if p)
    _q27_busy = []
    _q27_native = _q27_re.compile(
        r"^_*(open|fopen|freopen|fdopen|wopen|wfopen|creat|read|pread|fread|fgets|fgetc|getline|getdelim|fscanf|mmap|"
        r"syscall|system|popen|wpopen|exec|fexecve|fork|vfork|clone|posix_spawn|dlopen|dlmopen|dlsym|sendfile|splice|"
        r"copy_file_range|opendir|fdopendir|scandir|readlink|createfile|createprocess|winexec|shellexecute|"
        r"loadlibrary|ntcreatefile|ntopenfile|ntreadfile)", _q27_re.I)
    def _q27_hook(event, args, _deny=_q27_deny, _busy=_q27_busy, _native=_q27_native):
        if event in ("ctypes.dlsym", "ctypes.dlsym/handle"):
            _n = str(args[1]) if len(args) > 1 else ""
            if _native.match(_n):
                raise PermissionError("[sandbox] not allowed: native " + _n)
            return
        if event == "import" and args and str(args[0]).split(".")[0] in ("cffi", "_cffi_backend"):
            raise ImportError("[sandbox] not allowed: " + str(args[0]))
        if event in ("open", "os.listdir", "os.scandir", "os.chdir", "os.rename", "os.remove", "shutil.copyfile",
                     "shutil.copytree", "glob.glob", "os.walk", "os.symlink", "os.link", "sqlite3.connect",
                     "ctypes.dlopen"):
            if _busy:
                return
            _busy.append(1)
            try:                        # the path as given AND resolved (relative paths after chdir, symlinks)
                _ps = []
                for _a in args[:2]:
                    if isinstance(_a, (str, bytes)) or hasattr(_a, "__fspath__"):
                        _ps.append(str(_a))
                        try:
                            _ps.append(str(_q27_os2.path.realpath(_a)))
                        except Exception:
                            pass
            finally:
                _busy.pop()
            for _p in _ps:
                if any(d in _p.lower().replace("\\", "/") for d in _deny):
                    raise PermissionError("[sandbox] access denied: " + _p)
        elif event in ("subprocess.Popen", "os.system", "os.exec", "os.spawn", "os.posix_spawn", "os.fork",
                       "socket.connect"):
            raise PermissionError("[sandbox] not allowed: " + event)
    _q27_sys.addaudithook(_q27_hook)
    try:                                # scipy imports subprocess, so the module stays; its raw fork_exec does not
        import _posixsubprocess as _q27_ps
        def _q27_no_fork(*a, **k):
            raise PermissionError("[sandbox] not allowed: fork_exec")
        _q27_ps.fork_exec = _q27_no_fork
    except ImportError:
        pass
    try:                                # canary: the hook must really deny a forbidden path, else stop here
        open("/" + _q27_deny[0].strip("/") + "/__q27_canary__")
        _q27_ok = False
    except PermissionError:
        _q27_ok = True
    except OSError:
        _q27_ok = False
    if not _q27_ok:
        raise RuntimeError("audit hook inactive")
except BaseException as _q27_e:
    import sys as _q27_sys2
    _q27_sys2.stderr.write("[sandbox] guard failed, refusing to run: %r\n" % (_q27_e,))
    _q27_sys2.stderr.flush()
    _q27_sys2.exit(3)
import sys, json, io, contextlib, copy, traceback
MARK = "@@ARC_RESULT@@"
job = json.loads(sys.stdin.read())

def clean(res):
    try:
        import numpy as np
        if isinstance(res, np.ndarray):
            res = res.tolist()
    except Exception:
        pass
    if isinstance(res, tuple):
        res = list(res)
    if not isinstance(res, list) or not res:
        return None, "output is not a non-empty list of rows"
    rows = []
    for row in res:
        if hasattr(row, "tolist"):
            row = row.tolist()
        if not isinstance(row, (list, tuple)) or not row:
            return None, "output rows must be non-empty lists"
        out = []
        for v in row:
            try:
                iv = int(v)
            except Exception:
                return None, f"non-integer cell {v!r}"
            if iv != v or not 0 <= iv <= 9:
                return None, f"cell value {v!r} outside 0-9"
            out.append(iv)
        rows.append(out)
    w = len(rows[0])
    if any(len(r) != w for r in rows):
        return None, "output is not rectangular"
    if not (1 <= len(rows) <= 30 and 1 <= w <= 30):
        return None, f"output size {len(rows)}x{w} outside 1..30"
    return rows, None

results = []
sink = io.StringIO()
ns = {"__name__": "__arc_program__"}
try:
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
        exec(compile(job["code"], "<program>", "exec"), ns)
    fn = ns.get("transform")
    if not callable(fn):
        raise NameError("no function named transform")
except Exception as e:
    msg = "".join(traceback.format_exception_only(type(e), e)).strip()
    results = [{"error": "program failed to load: " + msg[-400:]} for _ in job["inputs"]]
else:
    for grid in job["inputs"]:
        try:
            with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
                res = fn(copy.deepcopy(grid))
            rows, err = clean(res)
            results.append({"output": rows} if err is None else {"error": err})
        except Exception as e:
            tb = traceback.format_exc().strip().splitlines()
            results.append({"error": " | ".join(tb[-3:])[-400:]})
sys.stdout.write(MARK + json.dumps(results) + "\n")
sys.stdout.flush()
'''


def _limits(mem_bytes):
    def apply():
        try:
            import resource
            resource.setrlimit(resource.RLIMIT_AS, (mem_bytes, mem_bytes))
        except Exception:
            pass
    return apply if os.name == "posix" else None


def run_one(code, inputs, timeout=10.0, mem_bytes=2 << 30):
    """Run one program on a list of input grids; never raises."""
    try:
        p = subprocess.run(
            [sys.executable, "-c", RUNNER],
            input=json.dumps({"code": code, "inputs": inputs}),
            capture_output=True, text=True, timeout=timeout,
            env=dict(os.environ, OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1",
                     MKL_NUM_THREADS="1", PYTHONHASHSEED="0", Q27_CHILD_MAX_AS=str(mem_bytes)))   # v1.4
    except subprocess.TimeoutExpired:
        return [{"error": f"timed out after {timeout:.0f}s"} for _ in inputs]
    for line in reversed(p.stdout.splitlines()):
        if line.startswith(MARK):
            try:
                res = json.loads(line[len(MARK):])
                if len(res) == len(inputs):
                    return res
            except Exception:
                break
    tail = (p.stderr or "").strip().splitlines()[-2:]
    why = " | ".join(tail)[-300:] or f"exit code {p.returncode}"
    return [{"error": "program crashed: " + why} for _ in inputs]


def run_many(jobs, timeout=10.0, workers=None, mem_bytes=2 << 30):
    """jobs: list of (code, inputs). Returns results in the same order."""
    workers = workers or max(1, (os.cpu_count() or 2) - 1)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(run_one, c, i, timeout, mem_bytes) for c, i in jobs]
        return [f.result() for f in futs]


In [ ]:
%%writefile /kaggle/working/agent/backend.py
"""LLM backends. The agent only needs one call: complete a batch of chats.

`VLLMBackend` runs offline on the Kaggle GPUs. `MockBackend` replays scripted
programs so the whole harness can be exercised on CPU with no model.
"""
import re
import time

CODE_RE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.S)
THINK_RE = re.compile(r"<think>.*?</think>", re.S)


def extract_code(text):
    """The last fenced block that defines transform(); falls back to any block
    containing it, then to a bare `def transform` in unfenced text."""
    if not text:
        return None
    body = THINK_RE.sub("", text)
    if "<think>" in body and "</think>" not in body:
        return None                      # truncated inside the reasoning
    blocks = [b for b in CODE_RE.findall(body) if "def transform" in b]
    if blocks:
        return blocks[-1].strip()
    i = body.rfind("def transform")
    if i >= 0:                           # unfenced: take the function to the end
        return "import numpy as np\n" + body[i:].strip()
    return None


class VLLMBackend:
    """Offline vLLM chat. One engine over all GPUs (tensor parallel)."""

    def __init__(self, model, tp=4, max_model_len=32768, gpu_mem=0.90,
                 max_tokens=16384, temperature=0.8, top_p=0.95, seed=0,
                 chat_kwargs=None, quantization=None, enforce_eager=False):
        from vllm import LLM, SamplingParams
        self.SamplingParams = SamplingParams
        self.llm = LLM(model=model, tensor_parallel_size=tp, max_model_len=max_model_len,
                       gpu_memory_utilization=gpu_mem, seed=seed, trust_remote_code=True,
                       quantization=quantization, enforce_eager=enforce_eager)
        self.max_tokens, self.temperature, self.top_p = max_tokens, temperature, top_p
        self.chat_kwargs = chat_kwargs or {}
        self.seed = seed
        self.tokenizer = self.llm.get_tokenizer()

    def prompt_tokens(self, messages):
        ids = self.tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                                 **self.chat_kwargs)
        return len(ids)

    def chat(self, conversations, max_tokens=None, metas=None):
        """Returns [(text, n_generated_tokens, finish_reason)] in order."""
        params = [self.SamplingParams(temperature=self.temperature, top_p=self.top_p,
                                      max_tokens=max_tokens or self.max_tokens,
                                      seed=self.seed + i)
                  for i in range(len(conversations))]
        self.seed += len(conversations)
        outs = self.llm.chat(conversations, params, use_tqdm=False,
                             chat_template_kwargs=self.chat_kwargs)
        res = []
        for o in outs:
            c = o.outputs[0]
            res.append((c.text, len(c.token_ids), c.finish_reason))
        return res


class MockBackend:
    """Replays `script(task_id, round, attempt, conversation) -> text`.

    `metas` (task_id, attempt) travel beside the conversations, never inside
    them, so the vLLM path sees plain chat messages.
    """

    def __init__(self, script, delay=0.0):
        self.script = script
        self.delay = delay
        self.calls = 0

    def prompt_tokens(self, messages):
        return sum(len(m["content"]) for m in messages) // 3

    def chat(self, conversations, max_tokens=None, metas=None):
        self.calls += 1
        time.sleep(self.delay)
        out = []
        for conv, meta in zip(conversations, metas or [{}] * len(conversations)):
            rnd = sum(m["role"] == "assistant" for m in conv)
            text = self.script(meta.get("task_id"), rnd, meta.get("attempt", 0), conv)
            out.append((text, len(text) // 3, "stop"))
        return out


In [ ]:
%%writefile /kaggle/working/agent/agent.py
"""Batched program-synthesis agent for ARC tasks.

Every task gets K independent attempts (conversations). Each ROUND sends every
live conversation to the model in ONE batch -- that is what keeps the GPUs
busy -- then runs each extracted `transform` on the task's train and test inputs
in the sandbox. An attempt whose program reproduces every train output is
VERIFIED and stops. Otherwise the model is shown where its program failed and
asked again, up to R rounds.

The verification against the demos is the point of the whole approach: a
program that reproduces all examples is a far stronger selector than any
likelihood score (exp06: the NLL scorer prefers a wrong grid over the injected
truth for 83 % of missed outputs).

Answers per test output are voted over verified programs; if fewer than two
distinct verified answers exist, the gap is filled from the best unverified
programs (most train examples right), and the source of each answer is kept so
a fallback solver can be preferred over weak guesses.
"""
import time
from collections import Counter, defaultdict

from grids import SYSTEM, task_prompt, feedback_prompt
from backend import extract_code
from sandbox import run_many


def visible_reply(text, keep=6000):
    """What goes back into the conversation: the answer without the reasoning.

    Qwen3-style models wrap reasoning in <think>; gpt-oss via raw harmony output
    puts the answer after a 'final' channel marker. Either way the program is in
    the tail, and re-sending long reasoning would crowd out the context.
    """
    t = text or ""
    if "</think>" in t:
        t = t.split("</think>")[-1]
    for marker in ("<|channel|>final<|message|>", "assistantfinal"):
        if marker in t:
            t = t.split(marker)[-1]
    t = t.strip()
    return t[-keep:] if len(t) > keep else t


def _key(g):
    return tuple(tuple(r) for r in g)


class Attempt:
    __slots__ = ("task_id", "idx", "messages", "rounds", "status", "programs",
                 "tokens", "note")

    def __init__(self, task_id, idx, prompt):
        self.task_id, self.idx = task_id, idx
        self.messages = [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": prompt}]
        self.rounds = 0
        self.status = "active"          # active | verified | failed | dropped
        self.programs = []              # dicts: round, code, n_ok, n_train, test
        self.tokens = 0
        self.note = ""


def solve(tasks, backend, samples=4, rounds=3, max_tokens=None, deadline=None,
          stop_after=2, max_prompt_tokens=None, sandbox_timeout=10.0,
          sandbox_workers=None, log=print):
    """tasks: {task_id: {"train": [...], "test": [...]}}.

    Returns (answers, report). answers[task_id] is a list over test inputs of
    [(grid, source), ...] best first; report holds per-attempt diagnostics.
    """
    t0 = time.time()
    deadline = deadline or float("inf")
    attempts = [Attempt(tid, i, task_prompt(task))
                for tid, task in tasks.items() for i in range(samples)]
    verified_count = Counter()
    rounds_log = []

    for rnd in range(1, rounds + 1):
        live = [a for a in attempts if a.status == "active"
                and verified_count[a.task_id] < stop_after]
        for a in attempts:
            if a.status == "active" and verified_count[a.task_id] >= stop_after:
                a.status, a.note = "failed", "task already verified"
        if not live:
            break
        if time.time() > deadline:
            log(f"round {rnd}: deadline reached with {len(live)} attempts live")
            break
        if max_prompt_tokens:
            keep = []
            for a in live:
                if backend.prompt_tokens(a.messages) > max_prompt_tokens:
                    a.status, a.note = "dropped", "prompt too long"
                else:
                    keep.append(a)
            live = keep
            if not live:
                break

        tg = time.time()
        replies = backend.chat([a.messages for a in live], max_tokens=max_tokens,
                               metas=[{"task_id": a.task_id, "attempt": a.idx} for a in live])
        gen_s = time.time() - tg

        jobs, owners = [], []
        for a, (text, ntok, finish) in zip(live, replies):
            a.rounds += 1
            a.tokens += ntok
            code = extract_code(text)
            a.messages.append({"role": "assistant", "content": visible_reply(text)})
            if code is None:
                why = ("Your reply was cut off before the program." if finish == "length"
                       else "No ```python block defining transform() was found.")
                a.messages.append({"role": "user", "content":
                                   why + " Give the complete program in ONE ```python code block."})
                if a.rounds >= rounds:
                    a.status, a.note = "failed", "no code"
                continue
            task = tasks[a.task_id]
            inputs = [p["input"] for p in task["train"]] + [t["input"] for t in task["test"]]
            jobs.append((code, inputs))
            owners.append((a, code))

        ts = time.time()
        results = run_many(jobs, timeout=sandbox_timeout, workers=sandbox_workers)
        run_s = time.time() - ts

        n_ver = 0
        for (a, code), res in zip(owners, results):
            task = tasks[a.task_id]
            ntr = len(task["train"])
            train_res = []
            for r, pair in zip(res[:ntr], task["train"]):
                ok = "output" in r and r["output"] == pair["output"]
                train_res.append({"ok": ok, "output": r.get("output"), "error": r.get("error")})
            n_ok = sum(t["ok"] for t in train_res)
            test_out = [r.get("output") for r in res[ntr:]]
            a.programs.append({"round": a.rounds, "code": code, "n_ok": n_ok,
                               "n_train": ntr, "test": test_out,
                               "test_errors": [r.get("error") for r in res[ntr:]]})
            if n_ok == ntr and all(o is not None for o in test_out):
                a.status = "verified"
                verified_count[a.task_id] += 1
                n_ver += 1
            elif a.rounds >= rounds:
                a.status, a.note = "failed", f"best {max(p['n_ok'] for p in a.programs)}/{ntr}"
            else:
                a.messages.append({"role": "user",
                                   "content": feedback_prompt(train_res, task["train"])})

        toks = sum(r[1] for r in replies)
        rounds_log.append({"round": rnd, "live": len(live), "programs": len(jobs),
                           "verified_now": n_ver, "gen_s": gen_s, "run_s": run_s,
                           "tokens": toks, "tok_per_s": toks / max(gen_s, 1e-9)})
        log(f"round {rnd}: {len(live)} live, {len(jobs)} programs, {n_ver} newly verified, "
            f"tasks verified {sum(1 for t in tasks if verified_count[t])}/{len(tasks)}, "
            f"gen {gen_s:.0f}s ({toks / max(gen_s, 1e-9):.0f} tok/s) run {run_s:.1f}s")

    answers = {tid: aggregate(tid, tasks[tid], attempts) for tid in tasks}
    report = {
        "rounds": rounds_log,
        "wall_s": time.time() - t0,
        "attempts": [{"task_id": a.task_id, "idx": a.idx, "status": a.status,
                      "rounds": a.rounds, "tokens": a.tokens, "note": a.note,
                      "best_ok": max((p["n_ok"] for p in a.programs), default=-1),
                      "n_train": len(tasks[a.task_id]["train"]),
                      "programs": a.programs} for a in attempts],
    }
    return answers, report


def aggregate(tid, task, attempts):
    """Per test input: candidate grids best first, each tagged with its source.

    Verified programs vote; unverified programs are ranked by train examples
    solved, then by how many programs agree.
    """
    mine = [a for a in attempts if a.task_id == tid]
    out = []
    for j in range(len(task["test"])):
        ver = Counter()
        unv = defaultdict(lambda: [0, 0])          # key -> [best n_ok, votes]
        grids = {}
        for a in mine:
            for p in a.programs:
                g = p["test"][j] if j < len(p["test"]) else None
                if g is None:
                    continue
                k = _key(g)
                grids[k] = g
                if p["n_ok"] == p["n_train"]:
                    ver[k] += 1
                else:
                    u = unv[k]
                    u[0] = max(u[0], p["n_ok"])
                    u[1] += 1
        ranked = [(grids[k], f"verified x{n}") for k, n in ver.most_common()]
        seen = {k for k in ver}
        rest = sorted((k for k in unv if k not in seen),
                      key=lambda k: (-unv[k][0], -unv[k][1]))
        ranked += [(grids[k], f"unverified {unv[k][0]}/{len(task['train'])}") for k in rest]
        out.append(ranked)
    return out


def to_submission(answers, tasks, fallback=None):
    """Kaggle format. `fallback(task_id, j)` supplies grids when the agent has
    fewer than two; otherwise the test input is echoed."""
    sub = {}
    for tid, task in tasks.items():
        rows = []
        for j, t in enumerate(task["test"]):
            cands = [g for g, _ in answers.get(tid, [[]] * len(task["test"]))[j]]
            if fallback and len(cands) < 2:
                for g in fallback(tid, j):
                    if g not in cands:
                        cands.append(g)
            while len(cands) < 2:
                cands.append(t["input"])
            rows.append({"attempt_1": cands[0], "attempt_2": cands[1]})
        sub[tid] = rows
    return sub


def score(sub, solutions):
    """ARC Prize scoring: a task scores the fraction of its test outputs matched
    by either attempt; the total is the sum over tasks (and / n as a percent)."""
    total, per = 0.0, {}
    for tid, sols in solutions.items():
        rows = sub.get(tid, [])
        hit = 0
        for j, sol in enumerate(sols):
            if j < len(rows) and sol in (rows[j]["attempt_1"], rows[j]["attempt_2"]):
                hit += 1
        per[tid] = hit / len(sols)
        total += per[tid]
    return total, 100.0 * total / max(len(solutions), 1), per


In [ ]:
%%writefile /kaggle/working/agent/repl.py
"""The `python` tool: run an exploratory snippet with the task preloaded.

Each call is a fresh subprocess (nothing persists between calls, as in the
Tufa duck harness), so a crash or an infinite loop costs one call. The task is
injected as numpy arrays plus a few ARC helpers; stdout is captured and capped.

Optional function retention (dfranzen's ARC-AGI-3 fork of the duck harness): with a
store, plain top-level functions a call defines are kept as SOURCE and replayed into
later calls; variables never persist. Old transcripts show why: 30-38% of all
function code the model wrote re-defined a function it had already written, and
19 of 20 ab3q36 submits re-typed functions first written in a python call.
"""
import json
import os
import subprocess
import sys

HELPERS = r'''
import numpy as np
from collections import Counter, defaultdict, deque
import itertools, math, functools, copy, re

def show(g):
    """Digit rows, one line per row."""
    return "\n".join("".join(str(int(v)) for v in row) for row in np.asarray(g))

def diff(a, b):
    """Cells where a and b differ, as (row, col, a_value, b_value); size mismatch -> string."""
    a, b = np.asarray(a), np.asarray(b)
    if a.shape != b.shape:
        return f"shape mismatch {a.shape} vs {b.shape}"
    return [(int(r), int(c), int(a[r, c]), int(b[r, c])) for r, c in zip(*np.nonzero(a != b))]

def objects(g, bg=None, diagonal=False, multicolor=False):
    """Connected components. bg defaults to the most common colour.
    Returns dicts: color (or colors), cells [(r,c)], bbox (r0,c0,r1,c1), size, shape (cropped mask)."""
    g = np.asarray(g)
    if bg is None:
        vals, counts = np.unique(g, return_counts=True)
        bg = int(vals[np.argmax(counts)])
    H, W = g.shape
    seen = np.zeros_like(g, dtype=bool)
    nbrs = [(-1,0),(1,0),(0,-1),(0,1)] + ([(-1,-1),(-1,1),(1,-1),(1,1)] if diagonal else [])
    out = []
    for r in range(H):
        for c in range(W):
            if seen[r, c] or g[r, c] == bg:
                continue
            col = g[r, c]
            q, cells = deque([(r, c)]), []
            seen[r, c] = True
            while q:
                y, x = q.popleft(); cells.append((y, x))
                for dy, dx in nbrs:
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] and g[ny, nx] != bg \
                            and (multicolor or g[ny, nx] == col):
                        seen[ny, nx] = True; q.append((ny, nx))
            ys, xs = zip(*cells)
            r0, c0, r1, c1 = min(ys), min(xs), max(ys), max(xs)
            mask = np.zeros((r1 - r0 + 1, c1 - c0 + 1), dtype=int)
            for y, x in cells:
                mask[y - r0, x - c0] = g[y, x]
            d = {"cells": cells, "bbox": (r0, c0, r1, c1), "size": len(cells), "shape": mask}
            if multicolor:
                d["colors"] = sorted({int(g[y, x]) for y, x in cells})
            else:
                d["color"] = int(col)
            out.append(d)
    return out
'''

RUNNER = r'''
try:                                    # v1.4: memory cap set by the child itself (no preexec_fn)
    import os as _q27_os, resource as _q27_res
    _q27_lim = int(_q27_os.environ.get("Q27_CHILD_MAX_AS", "0"))
    if _q27_lim > 0:
        _q27_res.setrlimit(_q27_res.RLIMIT_AS, (_q27_lim, _q27_lim))
except Exception:
    pass
# v1.6 guard: a PEP 578 audit hook (cannot be removed by the program). v1.7 (GPT final audit): installation failure or
# an inactive hook now ABORTS the child (it was fail-open), native routes are narrowed (ctypes symbols for file /
# process functions, cffi, _posixsubprocess.fork_exec), sqlite3/ctypes.dlopen paths are checked. This is a best-effort
# guard for model-written code, NOT a security sandbox (Python documents audit hooks as unsuitable for that).
try:
    import sys as _q27_sys, os as _q27_os2, re as _q27_re
    _q27_deny = tuple(p for p in _q27_os2.environ.get("Q27_DENY_PATHS", "/kaggle/input|solution").lower().split("|") if p)
    _q27_busy = []
    _q27_native = _q27_re.compile(
        r"^_*(open|fopen|freopen|fdopen|wopen|wfopen|creat|read|pread|fread|fgets|fgetc|getline|getdelim|fscanf|mmap|"
        r"syscall|system|popen|wpopen|exec|fexecve|fork|vfork|clone|posix_spawn|dlopen|dlmopen|dlsym|sendfile|splice|"
        r"copy_file_range|opendir|fdopendir|scandir|readlink|createfile|createprocess|winexec|shellexecute|"
        r"loadlibrary|ntcreatefile|ntopenfile|ntreadfile)", _q27_re.I)
    def _q27_hook(event, args, _deny=_q27_deny, _busy=_q27_busy, _native=_q27_native):
        if event in ("ctypes.dlsym", "ctypes.dlsym/handle"):
            _n = str(args[1]) if len(args) > 1 else ""
            if _native.match(_n):
                raise PermissionError("[sandbox] not allowed: native " + _n)
            return
        if event == "import" and args and str(args[0]).split(".")[0] in ("cffi", "_cffi_backend"):
            raise ImportError("[sandbox] not allowed: " + str(args[0]))
        if event in ("open", "os.listdir", "os.scandir", "os.chdir", "os.rename", "os.remove", "shutil.copyfile",
                     "shutil.copytree", "glob.glob", "os.walk", "os.symlink", "os.link", "sqlite3.connect",
                     "ctypes.dlopen"):
            if _busy:
                return
            _busy.append(1)
            try:                        # the path as given AND resolved (relative paths after chdir, symlinks)
                _ps = []
                for _a in args[:2]:
                    if isinstance(_a, (str, bytes)) or hasattr(_a, "__fspath__"):
                        _ps.append(str(_a))
                        try:
                            _ps.append(str(_q27_os2.path.realpath(_a)))
                        except Exception:
                            pass
            finally:
                _busy.pop()
            for _p in _ps:
                if any(d in _p.lower().replace("\\", "/") for d in _deny):
                    raise PermissionError("[sandbox] access denied: " + _p)
        elif event in ("subprocess.Popen", "os.system", "os.exec", "os.spawn", "os.posix_spawn", "os.fork",
                       "socket.connect"):
            raise PermissionError("[sandbox] not allowed: " + event)
    _q27_sys.addaudithook(_q27_hook)
    try:                                # scipy imports subprocess, so the module stays; its raw fork_exec does not
        import _posixsubprocess as _q27_ps
        def _q27_no_fork(*a, **k):
            raise PermissionError("[sandbox] not allowed: fork_exec")
        _q27_ps.fork_exec = _q27_no_fork
    except ImportError:
        pass
    try:                                # canary: the hook must really deny a forbidden path, else stop here
        open("/" + _q27_deny[0].strip("/") + "/__q27_canary__")
        _q27_ok = False
    except PermissionError:
        _q27_ok = True
    except OSError:
        _q27_ok = False
    if not _q27_ok:
        raise RuntimeError("audit hook inactive")
except BaseException as _q27_e:
    import sys as _q27_sys2
    _q27_sys2.stderr.write("[sandbox] guard failed, refusing to run: %r\n" % (_q27_e,))
    _q27_sys2.stderr.flush()
    _q27_sys2.exit(3)
import sys, json, io, contextlib, traceback
job = json.loads(sys.stdin.read())
ns = {"__name__": "__arc_repl__"}
exec(job["helpers"], ns)
import numpy as np
ns["train_inputs"] = [np.array(g) for g in job["train_inputs"]]
ns["train_outputs"] = [np.array(g) for g in job["train_outputs"]]
ns["test_inputs"] = [np.array(g) for g in job["test_inputs"]]
ns["train"] = list(zip(ns["train_inputs"], ns["train_outputs"]))
keep = job.get("retained")                # None: retention off (each call starts fresh)
preloaded = set(ns)
restore_failed = []
if keep is not None:
    # Replay what earlier calls kept: their imports, then their function definitions.
    # Source only -- no variable or array is ever carried over.
    for src in keep["imports"]:
        try:
            exec(compile(src, "<kept>", "exec"), ns)
        except Exception:
            pass
    for name, src in keep["defs"].items():
        try:
            exec(compile(src, "<kept>", "exec"), ns)
        except Exception:
            restore_failed.append(name)
buf = io.StringIO()
try:
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        exec(compile(job["code"], "<snippet>", "exec"), ns)
        if "result" in ns and ns["result"] is not None:
            print(repr(ns["result"]))
except Exception:
    tb = traceback.format_exc().strip().splitlines()
    buf.write("\n" + "\n".join(tb[-4:]))
sys.stdout.write("@@REPL@@" + json.dumps(buf.getvalue()) + "\n")
if keep is not None:
    import ast, builtins, dis, types
    defs, imports, rejected = {}, [], {}
    try:
        tree = ast.parse(job["code"])
    except SyntaxError:
        tree = ast.Module(body=[], type_ignores=[])
    bound = set()
    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)) and not getattr(node, "level", 0):
            src = ast.get_source_segment(job["code"], node)
            if src:
                imports.append(src)
                bound |= {(a.asname or a.name).split(".")[0] for a in node.names}
    new = [n for n in tree.body if isinstance(n, ast.FunctionDef)]
    available = (set(dir(builtins)) | preloaded | set(keep["defs"]) | bound
                 | {n.name for n in new} | set(keep.get("bound", [])))
    for node in new:
        fn = ns.get(node.name)
        # only a def that actually ran and still holds the name (not reassigned later)
        if not (isinstance(fn, types.FunctionType) and fn.__code__.co_filename == "<snippet>"
                and fn.__code__.co_firstlineno == node.lineno):
            continue
        reads, todo = set(), [fn.__code__]
        while todo:
            co = todo.pop()
            reads |= {i.argval for i in dis.get_instructions(co)
                      if i.opname in ("LOAD_GLOBAL", "LOAD_NAME")}
            todo += [c for c in co.co_consts if isinstance(c, types.CodeType)]
        missing = sorted(r for r in reads if isinstance(r, str) and r not in available)
        if missing:
            rejected[node.name] = missing
        else:
            defs[node.name] = ast.get_source_segment(job["code"], node)
    sys.stdout.write("@@KEPT@@" + json.dumps({"defs": defs, "imports": imports, "bound": sorted(bound),
                                             "rejected": rejected,
                                             "restore_failed": restore_failed}) + "\n")
'''

PREAMBLE_DOC = (
    "Preloaded in every `python` call (each call starts fresh; nothing persists): "
    "`np`, `train_inputs`, `train_outputs`, `test_inputs` (lists of numpy int arrays), "
    "`train` (list of (input, output) pairs), and helpers `show(g)` (digit rows), "
    "`diff(a, b)` (differing cells as (row, col, a, b)), "
    "`objects(g, bg=None, diagonal=False, multicolor=False)` (connected components with "
    "color, cells, bbox, size, shape). Print compact summaries, never whole large grids."
)


RETAIN_DOC = (
    "Variables do not persist between calls, but every function you define with a plain "
    "top-level `def` is kept: later `python` calls and `submit` can call it without "
    "redefining it. A kept function may use builtins, the preloaded names, imports and "
    "other kept functions; pass anything else in as an argument.")


TOOL_ONLY = ("train_inputs", "train_outputs", "test_inputs", "train")


def tool_only_globals(src):
    """v1.7 (GPT final audit: `train` is preloaded too): names only the python tool preloads that `src` uses as
    GLOBALS. Scope-aware via symtable: a parameter, a function-local or a program-defined module-level `train` is
    fine; anything that would raise NameError in the submit sandbox is reported."""
    import symtable
    try:
        top = symtable.symtable(src, "<src>", "exec")
    except SyntaxError:
        return set()
    own = {s.get_name() for s in top.get_symbols() if s.is_assigned() or s.is_imported() or s.is_namespace()}
    bad, todo = set(), [top]
    while todo:
        t = todo.pop()
        for s in t.get_symbols():
            n = s.get_name()
            if n in TOOL_ONLY and n not in own and s.is_referenced() and s.is_global():
                bad.add(n)
        todo.extend(t.get_children())
    return bad


def new_store():
    """Per-attempt record of what python calls have kept (source only)."""
    return {"defs": {}, "imports": [], "bound": []}


def run_snippet(code, task, timeout=30.0, max_chars=4000, mem_bytes=2 << 30, store=None):
    """Run one snippet. With `store` (from new_store()), functions defined by earlier calls
    are replayed first and the ones this call defines are added to the store; a line at
    the end of the output says what was kept."""
    job = {"code": code, "helpers": HELPERS,
           "train_inputs": [p["input"] for p in task["train"]],
           "train_outputs": [p["output"] for p in task["train"]],
           "test_inputs": [t["input"] for t in task["test"]]}
    if store is not None:
        job["retained"] = store
    pre = None
    if os.name == "posix":
        def pre():
            try:
                import resource
                resource.setrlimit(resource.RLIMIT_AS, (mem_bytes, mem_bytes))
            except Exception:
                pass
    try:
        p = subprocess.run([sys.executable, "-c", RUNNER], input=json.dumps(job),
                           capture_output=True, text=True, timeout=timeout,
                           env=dict(os.environ, OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1",
                                    Q27_CHILD_MAX_AS=str(mem_bytes)))   # v1.4: no preexec_fn
    except subprocess.TimeoutExpired:
        return f"[timed out after {timeout:.0f}s]"
    out, kept = None, None
    for line in reversed(p.stdout.splitlines()):
        if line.startswith("@@KEPT@@") and kept is None:
            kept = json.loads(line[len("@@KEPT@@"):])
        if line.startswith("@@REPL@@"):
            out = json.loads(line[len("@@REPL@@"):])
            break
    if out is None:
        out = "[snippet crashed] " + " | ".join((p.stderr or "").strip().splitlines()[-2:])
    out = out.strip() or "[no output -- print() what you want to see]"
    if len(out) > max_chars:
        out = out[:max_chars] + f"\n... [truncated {len(out) - max_chars} chars]"
    if store is not None and kept is not None:
        out += _absorb(store, kept)
    return out


def _absorb(store, kept):
    """Merge one call's kept definitions into the store; return the note for the model."""
    for name in kept["restore_failed"]:
        store["defs"].pop(name, None)
    for src in kept["imports"]:
        if src not in store["imports"]:
            store["imports"].append(src)
    store["bound"] = sorted(set(store["bound"]) | set(kept["bound"]))
    notes = []
    for name, src in list(kept["defs"].items()):
        _bad = sorted(tool_only_globals(src))
        if _bad:                                     # v1.6: not portable to submit -> not kept (older copy dropped)
            kept["defs"].pop(name)
            store["defs"].pop(name, None)
            notes.append(f"not kept: {name} (uses {', '.join(_bad)}, which exist only in this tool -- "
                         "compute from the function's own arguments)")
    for name, src in kept["defs"].items():
        if store["defs"].get(name) != src:
            store["defs"].pop(name, None)            # re-insert: newest definition last
            store["defs"][name] = src
    if kept["defs"]:
        notes.append("kept for later calls: " + ", ".join(sorted(kept["defs"])))
    for name, missing in kept["rejected"].items():
        notes.append(f"not kept: {name} (uses {', '.join(missing[:4])} from this snippet -- "
                     "pass it as an argument to keep the function)")
    if kept["restore_failed"]:
        notes.append("dropped (failed to reload): " + ", ".join(kept["restore_failed"]))
    return ("\n[" + "; ".join(notes) + "]") if notes else ""


def with_kept(code, store):
    """A submitted program plus the kept functions (and imports) it relies on.

    Returns (full_source, names_added). The program's own definitions win; kept
    functions are added only if the program uses them, transitively, and transform()
    itself is taken from the store if the program does not define one."""
    import ast
    if not store or not store["defs"]:
        return code, []
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return code, []
    defined = set()
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
            defined.add(node.name)
        elif isinstance(node, (ast.Import, ast.ImportFrom)):
            defined |= {(a.asname or a.name).split(".")[0] for a in node.names}
        elif isinstance(node, ast.Assign):
            defined |= {t.id for t in node.targets if isinstance(t, ast.Name)}

    def used(src):
        try:
            return {n.id for n in ast.walk(ast.parse(src)) if isinstance(n, ast.Name)}
        except SyntaxError:
            return set()

    need = (used(code) & set(store["defs"])) - defined
    if "transform" not in defined and "transform" in store["defs"]:
        need.add("transform")
    todo = list(need)
    while todo:
        for n in (used(store["defs"][todo.pop()]) & set(store["defs"])) - defined - need:
            need.add(n)
            todo.append(n)
    if not need:
        return code, []
    # Kept functions were written against the python tool's namespace (np, objects(), ...),
    # so the helper preamble goes in first; the program's own definitions still win.
    parts = ([HELPERS.strip()] + list(store["imports"])
             + [src for n, src in store["defs"].items() if n in need])
    return "\n".join(parts) + "\n\n" + code, sorted(need)


In [ ]:
%%writefile /kaggle/working/agent/client.py
"""Minimal OpenAI-compatible chat client for a local vLLM server (stdlib only).

Returns a normalised dict: content, reasoning, tool_calls [{id, name, arguments}],
finish_reason, prompt_tokens, completion_tokens. If the server's tool parser
misses a call that the model wrote as qwen3_coder markup in its text, the call
is recovered from the markup, as the Tufa harness does.
"""
import json
import re
import time
import urllib.error
import urllib.request

TOOL_BLOCK = re.compile(r"<tool_call>\s*<function=([\w\-]+)>(.*?)</function>\s*</tool_call>", re.S)
PARAM = re.compile(r"<parameter=([\w\-]+)>\n?(.*?)\n?(?:</parameter>|\Z)", re.S)   # v1.1


def recover_markup(text):
    calls = []
    for name, body in TOOL_BLOCK.findall(text or ""):
        args = {k: v for k, v in PARAM.findall(body)}
        calls.append({"id": f"markup-{len(calls) + 1}", "name": name, "arguments": args})
    return calls


def strip_markup(text):
    return TOOL_BLOCK.sub("", text or "").strip()


class ChatClient:
    def __init__(self, base_url, model, temperature=0.6, top_p=0.95, top_k=20,
                 max_tokens=16384, thinking=True, timeout=1800, reasoning_effort=None,
                 thinking_token_budget=None, template_kwargs=None):
        self.url = base_url.rstrip("/") + "/chat/completions"
        self.model = model
        # top_k=None leaves it to the server (gpt-oss wants no top-k).
        self.sampling = {k: v for k, v in {"temperature": temperature, "top_p": top_p,
                                           "top_k": top_k}.items() if v is not None}
        self.reasoning_effort = reasoning_effort
        # vLLM >= 0.19 forces </think> after this many reasoning tokens (needs the server
        # flag --reasoning-config). Qwen3.8 otherwise spends whole 16k turns thinking.
        self.thinking_token_budget = thinking_token_budget
        # Extra chat-template kwargs, e.g. Qwen3.8's reasoning_effort (xhigh default /
        # medium / low) -- the template itself turns it into a system instruction.
        self.template_kwargs = template_kwargs or {}
        self.max_tokens = max_tokens
        self.thinking = thinking
        self.timeout = timeout

    def chat(self, messages, tools=None, max_tokens=None, seed=None, timeout=None,
             tool_choice=None, template_kwargs=None, _deadline=None):
        if _deadline is None:                          # v1.3: absolute deadline shared with the retry below
            _deadline = time.monotonic() + (timeout or self.timeout)
        _remaining = _deadline - time.monotonic()
        if _remaining <= 0:
            return {"error": "request deadline exhausted", "context_overflow": False, "timeout": True}
        # template_kwargs: per-request overrides on top of the client's own, e.g. the
        # reasoning-effort ladder stepping one turn down to "medium".
        payload = {"model": self.model, "messages": messages,
                   "max_tokens": max_tokens or self.max_tokens,
                   "chat_template_kwargs": {"enable_thinking": self.thinking, **self.template_kwargs,
                                            **(template_kwargs or {})},
                   **self.sampling}
        if seed is not None:
            payload["seed"] = seed
        if self.reasoning_effort:          # gpt-oss: low / medium / high
            payload["reasoning_effort"] = self.reasoning_effort
        if self.thinking_token_budget:
            payload["thinking_token_budget"] = self.thinking_token_budget
        if tools:
            payload["tools"] = tools
            # A named tool forces that call (vLLM guided decoding); default lets it choose.
            payload["tool_choice"] = ({"type": "function", "function": {"name": tool_choice}}
                                      if tool_choice else "auto")
        req = urllib.request.Request(self.url, data=json.dumps(payload).encode(),
                                     headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=_remaining) as r:
                body = json.loads(r.read().decode())
            return normalise(body)
        except urllib.error.HTTPError as e:
            detail = e.read().decode(errors="replace")[:500]
            overflow = "maximum context length" in detail or "context length" in detail
            if (e.code == 400 and not overflow and "thinking_token_budget" in payload
                    and any(w in detail.lower() for w in ("thinking_token_budget", "thinking budget", "token budget"))):
                # A server started without --reasoning-config (the last rungs of the startup
                # ladder) may refuse the budget: drop it for every later call, not fail them all.
                self.thinking_token_budget = None
                self.budget_dropped = True             # v1.3: reported by run_eval / smoke, printed once
                print("WARNING: server rejected thinking_token_budget -> dropped for all later calls", flush=True)
                return self.chat(messages, tools, max_tokens, seed, timeout, tool_choice, template_kwargs,
                                 _deadline=_deadline)
            return {"error": f"HTTP {e.code}: {detail}", "context_overflow": overflow}
        except Exception as e:
            # probe24 lost every result to one uncaught socket TimeoutError.
            text = f"{type(e).__name__}: {e}"
            return {"error": text[:300], "context_overflow": False,
                    "timeout": isinstance(e, TimeoutError) or "timed out" in text}


def normalise(body):
    ch = body["choices"][0]
    msg = ch.get("message", {}) or {}
    content = msg.get("content") or ""
    reasoning = msg.get("reasoning_content") or msg.get("reasoning") or ""
    calls = []
    for tc in msg.get("tool_calls") or []:
        fn = tc.get("function", {})
        try:
            args = json.loads(fn.get("arguments") or "{}")
        except json.JSONDecodeError:
            args = {"_raw": fn.get("arguments")}
        if not isinstance(args, dict):                 # v1.1: a list/str/number would crash .get() later
            args = {"_invalid_arguments": repr(args)[:300]}
        calls.append({"id": tc.get("id") or f"call-{len(calls) + 1}",
                      "name": fn.get("name"), "arguments": args})
    if not calls:
        calls = recover_markup(content) or recover_markup(reasoning)
        content = strip_markup(content)
    usage = body.get("usage") or {}
    return {"content": content, "reasoning": reasoning, "tool_calls": calls,
            "finish_reason": ch.get("finish_reason"),
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "completion_tokens": usage.get("completion_tokens", 0)}


In [ ]:
%%writefile /kaggle/working/agent/repl_agent.py
"""REPL tool agent for ARC-AGI-2 tasks, modelled on the Tufa Labs duck harness.

The model gets two tools:
  python(code)  explore the task: train/test grids are preloaded as numpy
                arrays with ARC helpers; stdout comes back, capped.
  submit(code)  a program defining transform(grid). It is run on every train
                input; the reply says which examples it reproduced and shows a
                cell diff for the rest. Reproducing ALL of them verifies the
                program, records its test outputs and ends the attempt.

Many attempts run concurrently against one vLLM OpenAI server, which batches
them. Old turns are evicted when the context budget is exceeded; the system
prompt and the task stay. Answers are aggregated exactly as in agent.py.

Optional, off by default, from dfranzen's ARC-AGI-3 fork of the duck harness (see
run_attempt): reasoning-effort ladder on cut turns, function retention, kept
reasoning, hysteresis trimming, repetition-loop guard, timeout-as-cut.
"""
import json
import threading
import time
import zlib
from concurrent.futures import ThreadPoolExecutor

from agent import aggregate
from backend import extract_code
from grids import SYSTEM, show, diff_text
from repl import run_snippet, PREAMBLE_DOC, RETAIN_DOC, new_store, with_kept
from sandbox import run_one

TOOLS = [
    {"type": "function", "function": {
        "name": "python",
        "description": "Run Python to inspect the task. " + PREAMBLE_DOC,
        "parameters": {"type": "object", "properties": {
            "code": {"type": "string", "description": "Python code; print() what you need."}},
            "required": ["code"]}}},
    {"type": "function", "function": {
        "name": "submit",
        "description": ("Submit a complete program defining transform(grid) -> grid. It is run on "
                        "every example input; you are told which examples it reproduces. "
                        "When it reproduces all of them it is accepted and you are done."),
        "parameters": {"type": "object", "properties": {
            "code": {"type": "string", "description": "Full Python source defining transform(grid)."}},
            "required": ["code"]}}},
]

REPL_INSTRUCTIONS = """Solve this ARC puzzle. The examples below share one hidden transformation rule.

Work like a scientist: use the `python` tool to measure the examples (objects, colours, sizes, positions, differences between input and output), form a hypothesis, and test it in code. Then call `submit` with the full source defining `transform(grid)`. Use only the standard library and numpy.

Submit EARLY and often: submitting is free, and a failing submission tells you exactly which examples and which cells are wrong -- that is the fastest way to find the rule. Do not wait for certainty. If a submission fails, study the diff and fix the rule rather than special-casing examples."""

# probe24: every failed attempt spent 30-73k tokens exploring and never submitted
# once, so it never saw a diff. After this many python calls without a submit,
# the tool output carries a reminder.
NUDGE_EVERY = 6
NUDGE = ("\n\n[harness] You have run {n} python calls without submitting. Submitting is free "
         "and returns a cell-level diff against every example. Submit your current best "
         "transform now, then refine it from the diff.")

# probe9q38: every failed attempt had 1-3 turns cut at 16k tokens mid-thought, while
# solves used 2-3k tokens a turn. The thinking-cap treatment lowers the per-turn cap
# and, when a turn is cut, orders the model to act instead of resuming the analysis.
THINKCAP_HINT = ("[harness] You hit the per-turn thinking limit. Stop analysing. In your next "
                 "reply, call `python` to test your current best hypothesis in code, or "
                 "`submit` a transform implementing it -- a failing submit shows exactly "
                 "which cells are wrong.")


# The probe24 wording, kept verbatim so a run with the nudge off matches it exactly.
REPL_INSTRUCTIONS_V1 = """Solve this ARC puzzle. The examples below share one hidden transformation rule.

Work like a scientist: use the `python` tool to measure the examples (objects, colours, sizes, positions, differences between input and output), form a hypothesis, and test it in code. When you have a program that you believe reproduces EVERY example, call `submit` with the full source defining `transform(grid)`. Use only the standard library and numpy. If a submission fails, study the diff and fix the rule rather than special-casing examples."""


# probe9q38 transcripts: failures parse and count grids IN THE REASONING, cell by cell,
# often filling a whole 16k-token turn before any tool call ("col 7=8, 8=0, 9=0 ...").
# Tufa's harness says the same thing to its model. Optional, its own arm.
IN_PYTHON_RULE = """

Never read, count, index or transcribe grid cells in your head. Do ALL of that with the `python` tool -- the grids are already loaded as numpy arrays. Keep your own reasoning to hypotheses and plans; let code do the looking."""


def repl_prompt(task, early_submit=True, in_python=False):
    head = REPL_INSTRUCTIONS if early_submit else REPL_INSTRUCTIONS_V1
    parts = [head + (IN_PYTHON_RULE if in_python else ""), ""]
    for i, p in enumerate(task["train"], 1):
        parts += [f"## Example {i}", show(p["input"], "Input"), show(p["output"], "Output"), ""]
    for j, t in enumerate(task["test"], 1):
        parts += [f"## Test input {j}", show(t["input"], "Input"), ""]
    return "\n".join(parts).rstrip()


ABORT = {"flag": False, "why": ""}                 # v1.6: shared stop switch (circuit breaker)
_BREAKER = {"n": 0, "fail": 0, "recent": []}
_BREAKER_LOCK = threading.Lock()


def _request_failure(rec):
    """an attempt that died on a request error (HTTP status, refused connection, ...) without one generated token"""
    return (rec.get("status") == "failed" and rec.get("gen_tokens", 0) == 0
            and str(rec.get("note", "")) not in ("", "token budget", "turn budget"))


def breaker(rec, first_n=6, window=12):
    """v1.6: call once per finished attempt; trips ABORT if the first `first_n` attempts all died on request errors
    with no generated tokens (a request-shape or server problem -- more attempts would only burn the GPU budget).
    v1.7 (GPT final audit): also if the last `window` finished attempts all did (the server died mid-run)."""
    with _BREAKER_LOCK:
        if ABORT["flag"]:
            return
        bad = _request_failure(rec)
        _BREAKER["recent"] = (_BREAKER["recent"] + [bad])[-window:]
        if _BREAKER["n"] < first_n:
            _BREAKER["n"] += 1
            _BREAKER["fail"] += int(bad)
            if _BREAKER["n"] == first_n and _BREAKER["fail"] == first_n:
                ABORT.update(flag=True, why=f"first {first_n} attempts failed on request errors: "
                                            f"{str(rec.get('note', ''))[:200]}")
        if not ABORT["flag"] and len(_BREAKER["recent"]) == window and all(_BREAKER["recent"]):
            ABORT.update(flag=True, why=f"last {window} attempts failed on request errors: "
                                        f"{str(rec.get('note', ''))[:200]}")
        if ABORT["flag"]:
            print("CIRCUIT BREAKER:", ABORT["why"], flush=True)


def _nonportable(code):
    """v1.6: names that exist only inside the python tool; a transform using them fails in the submit sandbox.
    v1.7: scope-aware (repl.tool_only_globals) and includes the preloaded `train`."""
    from repl import tool_only_globals
    return tool_only_globals(code)


def verify(code, task, timeout=10.0):
    """Run a submitted program; return (n_ok, test_outputs, feedback_text)."""
    tr = task["train"]
    _bad = sorted(_nonportable(code))
    if _bad:                                           # v1.6: fail fast with a precise instruction
        return 0, [None] * len(task["test"]), (
            "The program uses " + ", ".join(_bad) + ", which exist only inside the python tool. transform(grid) must "
            "compute everything from its `grid` argument (constants and its own helpers are fine). Rewrite and submit.")
    res = run_one(code, [p["input"] for p in tr] + [t["input"] for t in task["test"]],
                  timeout=timeout)
    n_ok, lines, shown = 0, [], 0
    for i, (r, p) in enumerate(zip(res[:len(tr)], tr), 1):
        if "output" in r and r["output"] == p["output"]:
            n_ok += 1
            continue
        if shown < 2:
            shown += 1
            lines.append(f"## Example {i} failed")
            lines.append(f"Error: {r['error']}" if "error" in r else diff_text(p["output"], r["output"]))
    test_out = [r.get("output") for r in res[len(tr):]]
    test_err = [r.get("error") for r in res[len(tr):] if "error" in r]
    if n_ok == len(tr):
        if test_err:
            return n_ok, test_out, ("Reproduces all examples but FAILS on a test input: "
                                    + test_err[0] + " -- fix it and submit again.")
        return n_ok, test_out, f"Accepted: reproduces all {len(tr)} examples."
    return n_ok, test_out, (f"Reproduced {n_ok} of {len(tr)} examples.\n" + "\n".join(lines)
                            + "\nFix the rule and submit again.")


def _est_tokens(messages):
    return len(json.dumps(messages)) // 3


def trim(messages, budget, keep_recent=6, drain=0, state=None):
    """Drop the oldest assistant/tool/user groups after the task prompt until the
    estimate fits. A group starts at an assistant message, so tool results are
    never separated from the call that produced them.

    With `drain` and a per-attempt `state` dict the cut has hysteresis (dfranzen's
    ARC-AGI-3 harness): it only moves when the estimate exceeds `budget`, and then
    drops down to `budget - drain`. Between those moments the retained history is
    byte-identical from turn to turn, so the server's prefix cache keeps serving it;
    without it, every turn past the budget shifts the cut by one group and
    re-prefills everything after the task prompt."""
    if drain and state is not None:
        return _trim_hysteresis(messages, budget, keep_recent, drain, state)
    head, rest = messages[:2], messages[2:]
    starts = [i for i, m in enumerate(rest) if m["role"] == "assistant"]
    dropped = 0
    while _est_tokens(head + rest) > budget and len(starts) > keep_recent:
        cut = starts[1]
        rest = rest[cut:]
        starts = [i - cut for i in starts[1:]]
        dropped += 1
    if dropped:
        note = {"role": "user", "content": f"[{dropped} older turns were removed to save context.]"}
        rest = [note] + rest
    return head + rest


def _trim_hysteresis(messages, budget, keep_recent, drain, state):
    head, rest = messages[:2], messages[2:]
    starts = [i for i, m in enumerate(rest) if m["role"] == "assistant"]
    max_cut = max(0, len(starts) - keep_recent)

    def view(cut):
        if cut <= 0:
            return head + rest
        note = {"role": "user", "content": f"[{cut} older turns were removed to save context.]"}
        return head + [note] + rest[starts[cut]:]

    cut = min(state.get("cut", 0), max_cut)
    if _est_tokens(view(cut)) > budget:
        while cut < max_cut and _est_tokens(view(cut)) > budget - drain:
            cut += 1
        state["evictions"] = state.get("evictions", 0) + 1
    state["cut"] = cut
    return view(cut)


def degenerate(text, min_chars=500, ratio=0.01):
    """A repetition loop rather than text: zlib compresses it below `ratio`. dfranzen saw a
    reasoning-only reply of 6,144 '!' under speculative decoding; prose compresses to
    0.2-0.3, an ASCII grid to ~0.02, so 0.01 only catches genuine loops."""
    if not text or len(text) < min_chars:
        return False
    raw = text.encode("utf-8", "replace")
    return len(zlib.compress(raw, 6)) / len(raw) < ratio


DEGENERATE_STUB = ("(previous reply was a repetition loop and has been omitted; do not continue "
                   "that pattern)")


def make_tools(retain=False):
    """The two tools; with function retention their descriptions say what persists."""
    if not retain:
        return TOOLS
    py = dict(TOOLS[0]["function"], description=(
        "Run Python to inspect the task. "
        + PREAMBLE_DOC.replace(" (each call starts fresh; nothing persists)", "") + " " + RETAIN_DOC))
    sub = dict(TOOLS[1]["function"], description=(
        TOOLS[1]["function"]["description"] + " The program may call functions kept from your "
        "python calls (they are added automatically); if it defines no transform(), the kept "
        "transform() is submitted."))
    return [{"type": "function", "function": py}, {"type": "function", "function": sub}]


class _Shared:
    def __init__(self):
        self.lock = threading.Lock()
        self.verified = {}


class Attempt:
    def __init__(self, task_id, idx):
        self.task_id, self.idx = task_id, idx
        self.programs, self.status, self.note = [], "active", ""
        self.turns = self.python_calls = self.submits = 0
        self.python_at_submit = self.nudges = self.cuts = self.forced = 0
        self.gen_tokens = self.prompt_tokens = 0
        # ARC-AGI-3 takeaways (all zero unless their option is on)
        self.effort_steps = self.timeouts = self.degenerate = self.kept_used = 0
        self.evictions = 0
        self.wall = 0.0
        # Everything the model thought, ran and saw -- so a failure can be read as
        # "right idea, out of budget" vs "never found the rule".
        self.transcript = []


TIMEOUT_HINT = ("[harness] Your previous reply took too long and was abandoned. Keep your "
                "reasoning short: call `python` to test your current best hypothesis, or `submit`.")


def run_attempt(att, task, client, shared, deadline, max_turns=40, max_gen_tokens=120000,
                context_budget=28000, stop_after=1, seed=0, snippet_timeout=30.0,
                nudge_every=NUDGE_EVERY, turn_max_tokens=None, in_python=False,
                force_submit_after=0, effort_ladder=None, retain_functions=False,
                keep_reasoning=False, reasoning_key="reasoning_content", keep_recent=6,
                trim_drain=0, degenerate_guard=False, timeout_as_cut=False):
    """One attempt at one task. The options after force_submit_after are the ARC-AGI-3
    (dfranzen / duck harness) takeaways, each off by default so the baseline is unchanged:

    effort_ladder     e.g. ["medium", "low"]: after a turn is cut at max_tokens (or times
                      out, with timeout_as_cut) the next turns run one reasoning_effort rung
                      lower; a submit that reproduces more examples than before restores
                      the client's own effort.
    retain_functions  python-defined functions persist (as source) into later python calls
                      and into submits that use them.
    keep_reasoning    send each completed turn's reasoning back under `reasoning_key`
                      (pair with chat-template kwarg preserve_thinking=True). Cut turns and
                      repetition loops are never sent back.
    trim_drain        hysteresis for the context trim (see trim()); keep_recent groups are
                      never evicted.
    degenerate_guard  a repetition-loop reply is replaced by a stub in the history.
                      Loops are COUNTED in every arm (att.degenerate).
    timeout_as_cut    a request timeout before the deadline is treated like a cut turn
                      (retry, one rung lower) instead of failing the attempt."""
    t0 = time.time()
    # The early-submit wording and the nudge are ONE treatment: off means probe24's harness.
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": repl_prompt(task, early_submit=bool(nudge_every),
                                                        in_python=in_python)}]
    tools = make_tools(retain_functions)
    store = new_store() if retain_functions else None
    trim_state = {} if trim_drain else None
    rung, best_ok = -1, -1                   # rung -1 = the client's own reasoning effort

    def step_down():
        nonlocal rung
        if effort_ladder and rung < len(effort_ladder) - 1:
            rung += 1
            att.effort_steps += 1

    while att.turns < max_turns:
        if ABORT["flag"]:                              # v1.6: circuit breaker tripped
            att.status, att.note = "stopped", "aborted: " + ABORT["why"][:150]
            break
        with shared.lock:
            if shared.verified.get(att.task_id, 0) >= stop_after:
                att.status, att.note = "stopped", "task verified by a sibling"
                break
        if time.time() > deadline:
            att.status, att.note = "stopped", "deadline"
            break
        if att.gen_tokens >= max_gen_tokens:
            att.status, att.note = "failed", "token budget"
            break
        att.turns += 1
        # A request may not outlive the deadline by more than a minute.
        req_timeout = max(60.0, min(3600.0, deadline - time.time() + 60.0))
        # mini2arm: the text nudge was read and ignored. The forcesubmit treatment
        # makes the NEXT call a submit after this many python calls without one.
        force = (force_submit_after
                 and att.python_calls - att.python_at_submit >= force_submit_after)
        if force:
            att.forced += 1
        extra = {}
        if rung >= 0:            # only then, so clients without the kwarg keep working
            extra["template_kwargs"] = {"reasoning_effort": effort_ladder[rung]}
        # v1.1: never ask for more than the attempt has left (the last turn could overshoot by 12k before)
        request_cap = min(turn_max_tokens or client.max_tokens, max_gen_tokens - att.gen_tokens)
        if request_cap < 256:
            att.status, att.note = "failed", "token budget"
            break
        r = client.chat(trim(messages, context_budget, keep_recent, trim_drain, trim_state),
                        tools=tools, seed=seed + att.turns,
                        timeout=req_timeout, max_tokens=request_cap,
                        tool_choice="submit" if force else None, **extra)
        if trim_state is not None:
            att.evictions = trim_state.get("evictions", 0)
        if "error" in r:
            if r.get("context_overflow") and keep_reasoning:
                # Kept reasoning can make even the protected recent turns too long for
                # the model; shed it (oldest first, newest last) before shrinking.
                held = [m for m in messages if reasoning_key in m]
                if held:
                    for m in held[:max(1, len(held) - 1)]:
                        m.pop(reasoning_key)
                    continue
            if r.get("context_overflow") and context_budget > 8000:
                context_budget = int(context_budget * 0.7)
                continue
            if time.time() >= deadline:
                # Requests are timed to end a minute after the deadline: this is the run
                # ending, not the attempt failing (ab3q36 logged 21 of these as failures).
                att.status, att.note = "stopped", "deadline (request cut)"
                break
            if timeout_as_cut and r.get("timeout"):
                att.timeouts += 1
                step_down()
                if messages[-1]["role"] != "user":
                    messages.append({"role": "user", "content": TIMEOUT_HINT})
                continue
            att.status, att.note = "failed", r["error"][:200]
            break
        att.gen_tokens += r["completion_tokens"]
        att.prompt_tokens += r["prompt_tokens"]
        cut = r["finish_reason"] == "length"
        loops = [k for k in ("reasoning", "content") if degenerate(r[k])]
        att.degenerate += bool(loops)
        att.transcript.append({"turn": att.turns, "t": round(time.time() - t0, 1),
                               "reasoning": r["reasoning"], "content": r["content"],
                               "calls": [c["name"] for c in r["tool_calls"]],
                               "finish": r["finish_reason"], "tokens": r["completion_tokens"],
                               **({"effort": effort_ladder[rung]} if rung >= 0 else {}),
                               **({"loop": loops} if loops else {})})
        if cut:
            step_down()
        content = r["content"] or ""
        if degenerate_guard and "content" in loops:
            content = DEGENERATE_STUB
        msg = {"role": "assistant", "content": content}
        if keep_reasoning and r["reasoning"] and not cut and "reasoning" not in loops:
            msg[reasoning_key] = r["reasoning"]
        calls = r["tool_calls"]
        if not calls:
            code = extract_code(r["content"])
            if code:                       # answered with a code block instead of submit
                calls = [{"id": f"implicit-{att.turns}", "name": "submit",
                          "arguments": {"code": code}, "implicit": True}]
            else:
                messages.append(msg)
                if r["finish_reason"] == "length":
                    # The thinking-cap treatment answers a cut with an order to act; the
                    # baseline keeps probe24's wording.
                    hint = THINKCAP_HINT if turn_max_tokens else "Your reply was cut off. Be more concise."
                    att.cuts += 1
                else:
                    hint = "Use the `python` tool to explore, or `submit` your transform program."
                messages.append({"role": "user", "content": hint})
                continue
        if not any(c.get("implicit") for c in calls):
            msg["tool_calls"] = [{"id": c["id"], "type": "function",
                                  "function": {"name": c["name"],
                                               "arguments": json.dumps(c["arguments"])}}
                                 for c in calls]
        messages.append(msg)
        done = False
        for c in calls:
            _args = c.get("arguments") if isinstance(c.get("arguments"), dict) else {}   # v1.1
            code = _args.get("code") if isinstance(_args.get("code"), str) else ""
            if c["name"] in ("python", "submit") and not code.strip():
                out = ("[harness] That tool call had no valid 'code' string. Call the tool again with the "
                       "complete Python source in the `code` argument.")
            elif c["name"] == "python":
                att.python_calls += 1
                out = run_snippet(code, task, timeout=snippet_timeout, store=store)
                since = att.python_calls - att.python_at_submit
                if nudge_every and since and since % nudge_every == 0:
                    out += NUDGE.format(n=since)
                    att.nudges += 1
            elif c["name"] == "submit":
                att.submits += 1
                att.python_at_submit = att.python_calls
                full, added = with_kept(code, store) if store else (code, [])
                n_ok, test_out, out = verify(full, task)
                if added:
                    att.kept_used += 1
                    out += f"\n(Ran with your kept functions: {', '.join(added)}.)"
                ntr = len(task["train"])
                att.programs.append({"turn": att.turns, "code": full, "n_ok": n_ok,
                                     "n_train": ntr, "test": test_out})
                if n_ok > best_ok:
                    if best_ok >= 0 or n_ok > 0:
                        rung = -1            # progress: back to the client's own effort
                    best_ok = n_ok
                if n_ok == ntr and all(o is not None for o in test_out):
                    done = True
            else:
                out = f"Unknown tool {c['name']!r}. Tools: python, submit."
            att.transcript.append({"turn": att.turns, "tool": c["name"], "code": code,
                                   "output": out})
            if c.get("implicit"):
                messages.append({"role": "user", "content": out})
            else:
                messages.append({"role": "tool", "tool_call_id": c["id"], "content": out})
        if done:
            att.status = "verified"
            with shared.lock:
                shared.verified[att.task_id] = shared.verified.get(att.task_id, 0) + 1
            break
    else:
        att.status, att.note = "failed", "turn budget"
    att.wall = time.time() - t0
    return att


# run_attempt options a probe or submission config may set for the whole run
# (probe arms can set any run_attempt keyword).
AGENT_OPTS = ("turn_max_tokens", "in_python", "force_submit_after", "effort_ladder",
              "retain_functions", "keep_reasoning", "reasoning_key", "keep_recent", "trim_drain",
              "degenerate_guard", "timeout_as_cut", "snippet_timeout", "attempt_caps")   # v1.6


def attempt_record(a, task):
    return {"task_id": a.task_id, "idx": a.idx, "status": a.status,
            "note": a.note, "turns": a.turns, "python_calls": a.python_calls,
            "submits": a.submits, "nudges": a.nudges, "cuts": a.cuts, "forced": a.forced, "gen_tokens": a.gen_tokens,
            "prompt_tokens": a.prompt_tokens, "wall": a.wall,
            "effort_steps": a.effort_steps, "timeouts": a.timeouts, "degenerate": a.degenerate,
            "kept_used": a.kept_used, "evictions": a.evictions,
            "best_ok": max((p["n_ok"] for p in a.programs), default=-1),
            "n_train": len(task["train"]),
            "programs": a.programs, "transcript": a.transcript}


def solve_repl(tasks, client, attempts=2, concurrency=32, deadline=None, stop_after=1,
               log=print, on_done=None, task_kw=None, **kw):
    """Same return shape as agent.solve: (answers, report).

    `on_done(record)` is called as each attempt finishes, so a caller can persist
    results incrementally -- probe24 lost everything to one late exception."""
    deadline = deadline or float("inf")
    shared = _Shared()
    # v1.1: attempt 0 of every task is queued first; attempt k+1 of a task is queued (at the back) only when its
    # attempt k ended unverified -- two attempts of one task never run at the same time
    atts = []
    t0 = time.time()
    done_n = [0]

    def work(att):
        try:
            # task_kw lets arms of an A/B share one run: key -> per-arm overrides. An arm
            # may bring its own client (different sampling / chat-template kwargs).
            over = dict(kw, **(task_kw or {}).get(att.task_id, {}))
            _caps = over.pop("attempt_caps", None)     # v1.6: e.g. [24000, 36000] = scout, then deep
            if _caps:
                over["max_gen_tokens"] = int(_caps[min(att.idx, len(_caps) - 1)])
            arm_client = over.pop("client", client)
            run_attempt(att, tasks[att.task_id], arm_client, shared, deadline,
                        stop_after=stop_after, seed=1000 * att.idx, **over)
        except Exception as e:                  # one attempt must never sink the run
            import traceback
            att.status, att.note = "crashed", f"{type(e).__name__}: {e}"[:300]
            att.transcript.append({"crash": traceback.format_exc()[-2000:]})
        if on_done:
            try:
                on_done(attempt_record(att, tasks[att.task_id]))
            except Exception as e:
                log(f"on_done failed: {e}")
        with shared.lock:
            done_n[0] += 1
            n_ver = sum(1 for v in shared.verified.values() if v)
        log(f"[{time.time() - t0:6.0f}s] {att.task_id}#{att.idx} {att.status:8s} "
            f"turns {att.turns:2d} py {att.python_calls:2d} sub {att.submits} "
            f"tok {att.gen_tokens:6d} | done {done_n[0]}/{len(atts)} tasks verified {n_ver}")
        return att

    from concurrent.futures import wait, FIRST_COMPLETED
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        live = set()
        for tid in tasks:
            a0 = Attempt(tid, 0)
            atts.append(a0)
            live.add(ex.submit(work, a0))
        while live:
            finished, live = wait(live, return_when=FIRST_COMPLETED)
            for f in finished:
                a = f.result()
                if (a.idx + 1 < attempts and time.time() < deadline
                        and shared.verified.get(a.task_id, 0) < stop_after):
                    nxt = Attempt(a.task_id, a.idx + 1)
                    atts.append(nxt)
                    live.add(ex.submit(work, nxt))

    answers = {tid: aggregate(tid, tasks[tid], atts) for tid in tasks}
    report = {"wall_s": time.time() - t0,
              "attempts": [attempt_record(a, tasks[a.task_id]) for a in atts]}
    return answers, report


In [ ]:
%%writefile /kaggle/working/agent/run_direct.py
"""Direct-answer mode: the model reasons (e.g. Qwen3.8 at reasoning_effort xhigh) and
writes the test output grid(s) itself -- no tools. Two samples per task give the two
attempts. This is how headline ARC-AGI-2 numbers for chat models are usually measured,
so it is the comparison the REPL agent has to beat.

    python run_direct.py '<json config>'

Config: base_url, model, task_ids, samples (2), concurrency, max_tokens, deadline_epoch,
temperature/top_p/top_k, template_kwargs, out, data_dir.
Writes <out> (JSON) and <out>.jsonl (one line per finished sample).
"""
import json
import os
import re
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from client import ChatClient                       # noqa: E402
from grids import show                              # noqa: E402

# grids.SYSTEM tells the model it solves puzzles "by writing Python programs" -- right for
# the REPL agent, wrong when the answer is the grid itself.
DIRECT_SYSTEM = (
    "You are an expert at solving ARC-AGI puzzles. Each puzzle has a few input->output "
    "example pairs that share one hidden transformation rule. Grids are 2D arrays of digits "
    "0-9 (colours); 0 is usually the background."
)

# A reply cut off before any grid (the thinking budget missing or ignored, or the answer
# itself cut) gets ONE short follow-up without thinking: the tail of its own reasoning,
# and an order to commit. An unanswered task scores nothing; a committed guess may score.
RESCUE = """{prompt}

## Your notes so far (they were cut off)
{notes}

Stop analysing now. Apply the rule you were converging on and give the final test output grid(s): each as rows of digits in its own ```grid code block, in test order, with nothing else in the block."""

DIRECT_INSTRUCTIONS = """Solve this ARC puzzle. The examples share one hidden transformation rule. Work out the rule, check it against every example, then apply it to the test input(s).

At the very end of your answer, give each test output grid as rows of digits inside its own ```grid code block, in test order, with nothing else in the block."""


def direct_prompt(task):
    parts = [DIRECT_INSTRUCTIONS, ""]
    for i, p in enumerate(task["train"], 1):
        parts += [f"## Example {i}", show(p["input"], "Input"), show(p["output"], "Output"), ""]
    for j, t in enumerate(task["test"], 1):
        parts += [f"## Test input {j}", show(t["input"], "Input"), ""]
    return "\n".join(parts).rstrip()


def parse_grids(text, n_tests):
    """The last n_tests digit-only code blocks (```grid preferred, any fence accepted)."""
    blocks = re.findall(r"```(?:grid)?[^\n]*\n(.*?)```", text or "", re.S)
    grids = []
    for b in blocks:
        rows = [r.strip() for r in b.strip().splitlines() if r.strip()]
        rows = [r.replace(" ", "").replace(",", "") for r in rows]
        if rows and all(r.isdigit() for r in rows) and len({len(r) for r in rows}) == 1:
            grids.append([[int(c) for c in r] for r in rows])
    return grids[-n_tests:] if len(grids) >= n_tests else None


def answer(client, task, seed, deadline, max_tokens=None, rescue_tokens=8192, notes_chars=16000):
    """One direct answer to `task`: reason, then write the test grid(s). Returns a record;
    rec["grids"] is the list of test grids, or None when nothing parseable came back."""
    n = len(task["test"])
    prompt = direct_prompt(task)

    def timeout():
        return max(60.0, min(4 * 3600, deadline - time.time() + 60))

    mt = max_tokens or client.max_tokens
    for _ in range(3):
        r = client.chat([{"role": "system", "content": DIRECT_SYSTEM},
                         {"role": "user", "content": prompt}],
                        seed=seed, max_tokens=mt, timeout=timeout())
        if not (r.get("context_overflow") and mt > 8192):
            break
        mt = int(mt * 0.75)                    # a long prompt: shrink the reply, never give up
    if "error" in r:
        return {"status": "error: " + r["error"][:200], "grids": None, "gen_tokens": 0}
    rec = {"finish": r["finish_reason"], "gen_tokens": r["completion_tokens"],
           "reasoning_chars": len(r["reasoning"] or ""),
           "content_tail": (r["content"] or "")[-1500:],
           "reasoning_tail": (r["reasoning"] or "")[-3000:]}
    grids = parse_grids(r["content"], n)
    if grids:
        return dict(rec, status="answered", grids=grids)
    notes = ((r["reasoning"] or "") + "\n" + (r["content"] or "")).strip()
    if notes and deadline - time.time() > 120:
        r2 = client.chat([{"role": "system", "content": DIRECT_SYSTEM},
                          {"role": "user", "content": RESCUE.format(prompt=prompt,
                                                                    notes=notes[-notes_chars:])}],
                         seed=seed + 1, max_tokens=rescue_tokens, timeout=timeout(),
                         template_kwargs={"enable_thinking": False, "reasoning_effort": "low"})
        if "error" in r2:
            rec["rescue_error"] = r2["error"][:200]
        else:
            rec["gen_tokens"] += r2["completion_tokens"]
            rec["rescue_finish"] = r2["finish_reason"]
            rec["rescue_tail"] = ((r2["content"] or "") or (r2["reasoning"] or ""))[-1500:]
            # Without thinking, a reasoning parser may still file the reply as reasoning.
            grids = parse_grids(r2["content"], n) or parse_grids(r2["reasoning"], n)
            if grids:
                return dict(rec, status="rescued", grids=grids)
    return dict(rec, status="no grid", grids=None)


def main(cfg):
    t0 = time.time()
    root = cfg.get("data_dir", "/kaggle/input")
    import glob
    ch = json.load(open(glob.glob(f"{root}/**/arc-agi_evaluation_challenges.json", recursive=True)[0]))
    sol = json.load(open(glob.glob(f"{root}/**/arc-agi_evaluation_solutions.json", recursive=True)[0]))
    ids = cfg["task_ids"]
    client = ChatClient(cfg["base_url"], cfg["model"], max_tokens=cfg.get("max_tokens", 49152),
                        temperature=cfg.get("temperature", 1.0), top_p=cfg.get("top_p", 0.95),
                        top_k=cfg.get("top_k", 20), thinking=cfg.get("thinking", True),
                        thinking_token_budget=cfg.get("thinking_token_budget"),
                        template_kwargs=cfg.get("template_kwargs"), timeout=4 * 3600)
    out = cfg.get("out", "/kaggle/working/direct_results.json")
    jsonl = out.replace(".json", ".jsonl")
    deadline = cfg.get("deadline_epoch", float("inf"))
    lock = threading.Lock()
    recs = []
    jobs = [(t, k) for k in range(cfg.get("samples", 2)) for t in ids]   # sample-major

    def work(job):
        t, k = job
        rec = {"task_id": t, "sample": k}
        if time.time() > deadline:
            rec["status"] = "skipped: deadline"
        else:
            ts = time.time()
            rec.update(answer(client, ch[t], 1000 * k + 7, deadline,
                              rescue_tokens=cfg.get("rescue_tokens", 8192)))
            rec["wall"] = round(time.time() - ts, 1)
            grids = rec["grids"]
            rec["correct"] = [g == s for g, s in zip(grids, sol[t])] if grids else None
        with lock:
            recs.append(rec)
            with open(jsonl, "a") as f:
                f.write(json.dumps(rec) + "\n")
            print(f"[{time.time() - t0:6.0f}s] {t}#{k} {rec['status']:10s} "
                  f"tok {rec.get('gen_tokens', 0):6d} finish {rec.get('finish')} "
                  f"correct {rec.get('correct')}", flush=True)

    with ThreadPoolExecutor(max_workers=cfg.get("concurrency", 16)) as ex:
        list(ex.map(work, jobs))

    # pass@2 per task: any test output matched by either sample (fractional per test).
    per = {}
    for t in ids:
        n = len(sol[t])
        hit = [False] * n
        for r in recs:
            if r["task_id"] == t and r.get("correct"):
                hit = [h or c for h, c in zip(hit, r["correct"])]
        per[t] = sum(hit) / n
    tok = [r["gen_tokens"] for r in recs if "gen_tokens" in r]
    summary = {"config": cfg, "per_task": per, "score": sum(per.values()), "n_tasks": len(ids),
               "samples": len(recs), "mean_gen_tokens": sum(tok) / max(len(tok), 1),
               "cut_at_max_tokens": sum(1 for r in recs if r.get("finish") == "length"),
               "wall_s": time.time() - t0, "records": recs}
    json.dump(summary, open(out, "w"))
    print(f"\nDIRECT SCORE {summary['score']:.2f} / {len(ids)}  mean tokens {summary['mean_gen_tokens']:.0f}  "
          f"cut {summary['cut_at_max_tokens']}  wall {summary['wall_s']:.0f}s", flush=True)
    print("per task:", {t: round(v, 2) for t, v in per.items()}, flush=True)


if __name__ == "__main__":
    main(json.loads(sys.argv[1]))


In [ ]:
%%writefile /kaggle/working/agent/run_submit.py
"""Competition run: the REPL agent on the hidden test set, writing submission.json.

    python run_submit.py '<json config>'

Safety first, because a crashed rerun wastes a day's submission:
  * the notebook writes a valid FALLBACK submission (each test input copied into
    both attempts) before anything heavy runs, via write_fallback();
  * as every attempt finishes, the submission is rewritten atomically with the
    best answers so far;
  * the agent stops at a hard deadline well inside the 12 h limit.

Budget: the per-attempt token cap is the estimated token supply (throughput x
time) divided by the number of hidden tasks, clamped -- the task count is only
known at run time (the public placeholder has 240).
"""
import glob
import json
import os
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from client import ChatClient                      # noqa: E402
from repl_agent import solve_repl, AGENT_OPTS, breaker   # noqa: E402

SUB = "/kaggle/working/submission.json"


def find(name, root="/kaggle/input"):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]


def write_atomic(obj, path=SUB):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f)
    os.replace(tmp, path)


def write_fallback(challenges, path=SUB):
    """A valid submission with no model at all: every attempt is the test input."""
    write_atomic({t: [{"attempt_1": x["input"], "attempt_2": x["input"]} for x in task["test"]]
                  for t, task in challenges.items()}, path)


def ranked_answers(records, task):
    """Per test input: candidate grids best first. Verified programs vote; then the
    unverified programs by train examples reproduced. Mirrors agent.aggregate."""
    out = []
    ntr = len(task["train"])
    for j in range(len(task["test"])):
        score = {}
        grids = {}
        for r in records:
            for p in r["programs"]:
                g = p["test"][j] if j < len(p["test"]) else None
                if g is None:
                    continue
                k = json.dumps(g)
                grids[k] = g
                verified = p["n_ok"] == ntr
                s = score.get(k, (0, 0, 0))
                score[k] = (s[0] + verified, max(s[1], p["n_ok"]), s[2] + 1)
        out.append([grids[k] for k in sorted(score, key=lambda k: score[k], reverse=True)])
    return out


def direct(cfg, tasks, client, deadline, sub, sub_path, t0):
    """Direct answers (run_direct.answer): every task's sample 0 before any sample 1, each
    task in `tasks` order. attempt_1 = the first answer; attempt_2 = the next DIFFERENT one."""
    from run_direct import answer
    lock = threading.Lock()
    got = {}
    jobs = [(t, k) for k in range(cfg.get("samples", 2)) for t in tasks]
    log = open(cfg.get("log_path", "/kaggle/working/direct_attempts.jsonl"), "a")
    count = {}

    def work(job):
        t, k = job
        if deadline - time.time() < cfg.get("min_start_s", 600):
            rec = {"status": "skipped: deadline", "grids": None, "gen_tokens": 0}
        else:
            ts = time.time()
            rec = answer(client, tasks[t], 1000 * k + 7, deadline, max_tokens=cfg.get("max_tokens"),
                         rescue_tokens=cfg.get("rescue_tokens", 8192))
            rec["wall"] = round(time.time() - ts, 1)
        with lock:
            if rec["grids"]:
                got.setdefault(t, []).append(rec["grids"])
                for j, x in enumerate(tasks[t]["test"]):
                    cands = []
                    for grids in got[t]:
                        if grids[j] not in cands:
                            cands.append(grids[j])
                    sub[t][j] = {"attempt_1": cands[0],
                                 "attempt_2": cands[1] if len(cands) > 1 else x["input"]}
                write_atomic(sub, sub_path)
            st = rec["status"].split(":")[0]
            count[st] = count.get(st, 0) + 1
            log.write(json.dumps(dict({k2: v for k2, v in rec.items() if k2 != "grids"},
                                      task_id=t, sample=k)) + "\n")
            log.flush()
            print(f"[{time.time() - t0:6.0f}s] {t}#{k} {rec['status'][:40]:12s} "
                  f"tok {rec.get('gen_tokens', 0):6d} finish {rec.get('finish')} | {count}", flush=True)

    with ThreadPoolExecutor(max_workers=cfg.get("concurrency", 10)) as ex:
        list(ex.map(work, jobs))
    log.close()
    print(f"done: {len(got)}/{len(tasks)} tasks answered, {count}, {time.time() - t0:.0f}s", flush=True)


def main(cfg):
    t0 = time.time()
    rerun = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
    sub_path = cfg.get("sub_path", SUB)
    ch = json.load(open(find(cfg.get("challenges", "arc-agi_test_challenges.json"),
                             cfg.get("data_dir", "/kaggle/input"))))
    if not rerun and not cfg.get("full_interactive"):
        # Interactive run (should not happen -- versions are Quick Saved): stay tiny.
        ch = {t: ch[t] for t in sorted(ch)[:cfg.get("interactive_tasks", 3)]}
    if not os.path.exists(sub_path):
        write_fallback(ch, sub_path)
    n = len(ch)
    deadline = cfg["deadline_epoch"]
    avail = max(0.0, deadline - time.time())
    supply = cfg.get("tok_per_s", 140) * avail
    cap = int(min(cfg.get("max_cap", 60000), max(cfg.get("min_cap", 15000), supply / max(n, 1) * cfg.get("cap_factor", 1.0))))   # v1.1: was x1.3
    print(f"rerun={rerun} tasks={n} available {avail / 3600:.2f} h, est supply {supply / 1e6:.1f}M "
          f"tokens -> per-attempt cap {cap}", flush=True)

    # Cheapest prompts first: if the deadline bites, the long tasks are the ones lost.
    order = sorted(ch, key=lambda t: len(json.dumps(ch[t])))
    tasks = {t: ch[t] for t in order}
    client = ChatClient(cfg["base_url"], cfg["model"], max_tokens=cfg.get("max_tokens", 16384),
                        temperature=cfg.get("temperature", 0.6), top_p=cfg.get("top_p", 0.95),
                        top_k=cfg.get("top_k", 20), thinking=cfg.get("thinking", True),
                        reasoning_effort=cfg.get("reasoning_effort"),
                        thinking_token_budget=cfg.get("thinking_token_budget"),
                        template_kwargs=cfg.get("template_kwargs"))

    sub = json.load(open(sub_path))
    if cfg.get("mode") == "direct":
        return direct(cfg, tasks, client, deadline, sub, sub_path, t0)

    from rescue import run_rescue, merge, direct_grid      # v1.6
    lock = threading.Lock()
    done = {}
    direct_recs = {}

    def tagged(tid):
        """ranked_answers with the verified flag (rescue.merge puts the direct answer after verified grids only)"""
        task, recs = tasks[tid], done.get(tid, [])
        ntr = len(task["train"])
        out = []
        for j, cands in enumerate(ranked_answers(recs, task)):
            ver = {json.dumps(p["test"][j]) for r in recs for p in r["programs"]
                   if p["n_ok"] == ntr and j < len(p["test"]) and p["test"][j] is not None}
            out.append([(g, "verified" if json.dumps(g) in ver else "unverified") for g in cands])
        return out

    def refresh(tid):                                  # caller holds `lock`
        task = tasks[tid]
        for j, ranked in enumerate(tagged(tid)):
            sub[tid][j] = merge(ranked, direct_grid(direct_recs.get(tid), j), task["test"][j]["input"])
        write_atomic(sub, sub_path)

    def on_done(rec):
        breaker(rec)                                   # v1.6: stop early if the first attempts all die on HTTP errors
        with lock:
            done.setdefault(rec["task_id"], []).append(rec)
            refresh(rec["task_id"])

    agent_deadline = (t0 + (deadline - t0) * float(cfg.get("agent_frac", 0.5))
                      if cfg.get("rescue") else deadline)
    if cfg.get("rerun_attempt_caps"):                  # v1.7: 240 tasks need a smaller scout than EVAL12 (GPT audit)
        cfg["attempt_caps"] = list(cfg["rerun_attempt_caps"])
    print(f"per-attempt caps {cfg.get('attempt_caps')}", flush=True)
    print(f"agent until {(agent_deadline - time.time()) / 3600:.2f} h from now, then direct rescue until "
          f"{(deadline - time.time()) / 3600:.2f} h", flush=True)
    _, report = solve_repl(
        tasks, client, attempts=cfg.get("attempts", 1), concurrency=cfg.get("concurrency", 24),
        deadline=agent_deadline, stop_after=1, max_turns=cfg.get("max_turns", 40), max_gen_tokens=cap,
        context_budget=cfg.get("context_budget", 24000), nudge_every=cfg.get("nudge_every", 0),
        turn_max_tokens=cfg.get("turn_max_tokens"), in_python=cfg.get("in_python", False),
        log=lambda m: print(m, flush=True), on_done=on_done,
        **{k: cfg[k] for k in AGENT_OPTS if k in cfg and k not in ("turn_max_tokens", "in_python")})

    ver = sum(1 for a in report["attempts"] if a["status"] == "verified")
    print(f"agent done: {ver}/{n} tasks verified, {sum(a['gen_tokens'] for a in report['attempts'])} "
          f"tokens, {time.time() - t0:.0f}s", flush=True)
    if cfg.get("rescue"):
        verified = {a["task_id"] for a in report["attempts"] if a["status"] == "verified"}
        # v1.7: tasks the agent never finished an attempt on come first (nothing at all to submit), then the
        # attempted-but-unverified ones; cheapest prompts first inside each group
        todo = ([t for t in tasks if t not in verified and t not in done]
                + [t for t in tasks if t not in verified and t in done])

        def on_result(tid, rec):                       # run_rescue calls this under its own lock
            with lock:
                direct_recs[tid] = rec
                refresh(tid)

        print(f"rescue (direct answers): {len(todo)} unverified tasks, {(deadline - time.time()) / 3600:.2f} h left",
              flush=True)
        recs, _ = run_rescue(todo, tasks, cfg, deadline, log=lambda m: print(m, flush=True), on_result=on_result)
        n_q = sum(len(r["grids"]) for r in recs.values())
        n_g = sum(1 for r in recs.values() for g in r["grids"] if g is not None)
        print(f"rescue done: {n_g}/{n_q} test inputs answered ({len(todo)} tasks), {time.time() - t0:.0f}s",
              flush=True)


if __name__ == "__main__":
    main(json.loads(sys.argv[1]))


In [ ]:
%%writefile /kaggle/working/agent/submit_main.py
"""Everything the competition rerun does after the fallback submission is written:
install the vLLM wheelhouse, start the server, run the agent. It NEVER raises --
an exception in a rerun fails the submission even when submission.json exists --
so every stage is caught and logged, and the fallback stands if a stage fails.

    python submit_main.py '<json config>'
"""
import glob
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
import urllib.request
from pathlib import Path

HERE = os.path.dirname(os.path.abspath(__file__))
SERVER_PID = Path("/kaggle/working/vllm_server.pid")     # v1.1: lets the launcher kill a leftover server group
REPORT_PATH = Path("/kaggle/working/agent_server_report.json")
LINK_DIR = Path("/tmp/q27a_cuda_link")
CUDA_LINK = {}
KERNEL_ISO = {}                                          # v1.7: unshare probe result (informational)
CURRENT = {"proc": None}


def ensure_libcuda():
    """v1.1 (Q27-XHIGH run 2: FlashInfer JIT failed with `ld: cannot find -lcuda`; Kaggle ships only libcuda.so.1).
    Make -lcuda linkable through a private symlink directory and prove it with a tiny link."""
    import shutil as _sh
    info = {"needed": True}
    cands = []
    for d in ("/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/local/cuda/lib64", "/usr/lib64"):
        cands += sorted(glob.glob(f"{d}/libcuda.so.1")) + sorted(glob.glob(f"{d}/libcuda.so.*[0-9]"))
    real = next((Path(c) for c in cands if Path(c).is_file()), None)
    if real is None:
        info.update(probe="SKIP", reason="libcuda.so.1 not found")
    else:
        LINK_DIR.mkdir(parents=True, exist_ok=True)
        link = LINK_DIR / "libcuda.so"
        if link.exists() or link.is_symlink():
            link.unlink()
        link.symlink_to(real.resolve())
        info.update(real=str(real), shim=str(link))
        cxx = _sh.which("c++") or _sh.which("g++") or _sh.which("gcc")
        if cxx:
            src = Path("/tmp/q27a_cuda_probe.c")
            src.write_text("int cuInit(unsigned int);\nint main(void){return 0;}\nvoid *p = (void *)cuInit;\n")
            env = dict(os.environ, LIBRARY_PATH=f"{LINK_DIR}:" + os.environ.get("LIBRARY_PATH", ""))
            r = subprocess.run([cxx, "-x", "c", str(src), "-o", "/tmp/q27a_cuda_probe", "-lcuda"],
                               capture_output=True, text=True, env=env, timeout=120)
            info.update(probe="PASS" if r.returncode == 0 else "FAIL", probe_out=(r.stdout + r.stderr)[-600:])
        else:
            info.update(probe="SKIP", reason="no C compiler")
    CUDA_LINK.clear()
    CUDA_LINK.update(info)
    print("libcuda link:", {k: v for k, v in info.items() if k != "probe_out"}, flush=True)
    return info


def _on_sigterm(signum, frame):
    """v1.1: a SIGTERM to this launcher also ends the vLLM server's process group (it runs in its own session)."""
    if CURRENT["proc"] is not None:
        try:
            os.killpg(CURRENT["proc"].pid, 9)
        except Exception:
            pass
    sys.exit(0)


def classify_startup_failure(path=None):
    """v1.3: why did a rung fail to come up? (read from the tail of the server log)"""
    try:
        tail = Path(path or LOG).read_text(errors="replace")[-20000:].lower()
    except Exception:
        return "NO_LOG"
    if "cannot find -lcuda" in tail or "cannot find libcuda" in tail:
        return "CUDA_LINKER"
    if "unrecognized arguments" in tail or "error: argument" in tail:
        return "UNSUPPORTED_FLAGS"
    if "out of memory" in tail or "outofmemory" in tail:
        return "OOM"
    return "UNKNOWN"


def _smoke_task():
    """v1.3: a fresh random identity task per smoke, so a hard-coded print cannot pass the python check."""
    import random
    h, w = random.choice([(2, 3), (3, 2), (2, 4), (4, 3), (3, 5), (5, 2)])
    grid = [[random.randint(1, 9) for _ in range(w)] for _ in range(h)]
    test = [[random.randint(1, 9) for _ in range(w)] for _ in range(h)]
    return {"train": [{"input": grid, "output": grid}], "test": [{"input": test}]}, f"({h}, {w})"


def smoke(cfg, smoke_deadline):
    """The server answering /v1/models does not prove tool calling. v1.3 checks, all inside ONE deadline:
      forced    a named `submit` call (thinking off)
      auto      an automatic `python` call (thinking off)
      thinking  an automatic `python` call with thinking on and the agent's REAL thinking_token_budget
      roundtrip the agent's real loop: python call -> tool result sent back (role "tool") -> second request ->
                a `submit` that verifies (no tool_choice, so the automatic submit path is tested too)
    A call only counts if its code really runs on a fresh random task (python must print the right shape, submit
    must verify and return the test grid). The thinking test fails if the client had to drop the budget.
    ok = (auto or thinking) and roundtrip."""
    sys.path.insert(0, HERE)
    from client import ChatClient
    from repl_agent import make_tools, verify
    from repl import run_snippet
    task, shape = _smoke_task()
    tools = make_tools(cfg.get("retain_functions", False))
    out = {"forced": [], "auto": [], "thinking": [], "roundtrip": [], "deadline_hit": False}

    def left():
        return smoke_deadline - time.time()

    def semantic(name, code):
        try:
            compile(code, "<smoke>", "exec")
        except Exception as e:
            return False, f"does not compile: {e}"[:200], None
        if left() <= 2:
            return False, "smoke deadline", None
        if name == "submit":
            n_ok, test_out, fb = verify(code, task, timeout=min(8.0, left()))
            return (n_ok == 1 and test_out == [task["test"][0]["input"]]), fb[:200], None
        o = run_snippet(code, task, timeout=min(8.0, left()))
        return (shape in o and "Traceback" not in o and "[snippet crashed]" not in o), o[:200], o

    def check(r, name):
        calls = r.get("tool_calls") or []
        if "error" in r or not calls or calls[0].get("name") != name:
            return False, "no tool call, or the wrong tool", None
        args = calls[0].get("arguments")
        code = args.get("code") if isinstance(args, dict) else None
        if not isinstance(code, str) or not code.strip():
            return False, "no code string", None
        return semantic(name, code)

    def rec(r, ok, why, t):
        return {"ok": ok, "why": why, "sec": round(time.time() - t, 1), "error": r.get("error"),
                "finish": r.get("finish_reason"), "completion_tokens": r.get("completion_tokens"),
                "calls": [(c.get("name"), str(c.get("arguments"))[:200]) for c in r.get("tool_calls") or []],
                "content": (r.get("content") or "")[:200]}

    def ask(cl, msgs, choice):
        rem = left()
        if rem <= 5:
            out["deadline_hit"] = True
            return None
        return cl.chat(msgs, tools=tools, tool_choice=choice, timeout=min(75.0, rem))

    sysmsg = {"role": "system", "content": "You are testing tool calling. Answer only with the requested tool call."}
    ask_submit = {"role": "user", "content": "Call the `submit` tool with a Python program that defines "
                                             "transform(grid) and returns grid unchanged."}
    ask_python = {"role": "user", "content": "Call the `python` tool with code that prints train_inputs[0].shape."}
    budget = cfg.get("thinking_token_budget") or 6144           # v1.3: the agent's real budget, not 1024
    plain = ChatClient("http://127.0.0.1:1234/v1", "qwen", temperature=0.0, max_tokens=1024, thinking=False)
    # effort "low" keeps the smoke short; what is tested is that the server ACCEPTS this budget value with tools
    think = ChatClient("http://127.0.0.1:1234/v1", "qwen", temperature=0.6,
                       max_tokens=max(int(cfg.get("turn_max_tokens") or 8192), budget + 1024), thinking=True,
                       thinking_token_budget=budget, template_kwargs={"reasoning_effort": "low"})
    first_python = {}
    for key, cl, msgs, name, choice in (("forced", plain, [sysmsg, ask_submit], "submit", "submit"),
                                        ("auto", plain, [sysmsg, ask_python], "python", None),
                                        ("thinking", think, [sysmsg, ask_python], "python", None)):
        for _ in range(2):
            t = time.time()
            r = ask(cl, msgs, choice)
            if r is None:
                break
            ok, why, tool_out = check(r, name)
            out[key].append(rec(r, ok, why, t))
            if ok:
                if name == "python":
                    first_python.setdefault(key, (cl, msgs, r, tool_out))
                break
    out["forced_ok"] = any(x["ok"] for x in out["forced"])
    out["auto_ok"] = any(x["ok"] for x in out["auto"])
    out["thinking_budget_requested"] = budget
    out["thinking_budget_retained"] = think.thinking_token_budget == budget
    out["thinking_ok"] = any(x["ok"] for x in out["thinking"]) and out["thinking_budget_retained"]
    # v1.5: the EXACT request shape the agent will send (its template_kwargs, sampling, budget, max_tokens, tools),
    # cut to 32 output tokens -- the server only has to ACCEPT it (run 1: the template rejected effort "high")
    agent_cl = ChatClient("http://127.0.0.1:1234/v1", "qwen", temperature=cfg.get("temperature", 0.6),
                          top_p=cfg.get("top_p", 0.95), top_k=cfg.get("top_k", 20), max_tokens=32,
                          thinking=cfg.get("thinking", True) and out["thinking_ok"],
                          thinking_token_budget=cfg.get("thinking_token_budget") if out["thinking_ok"] else None,
                          template_kwargs=cfg.get("template_kwargs"))   # same fallback the ladder will apply
    out["agent_params"] = []
    rungs_eff = [None] + [{"reasoning_effort": e} for e in (cfg.get("effort_ladder") or [])]
    for extra_kw in rungs_eff:                      # the agent's own effort and every effort-ladder step
        if left() <= 5:
            out["deadline_hit"] = True
            break
        t = time.time()
        r = agent_cl.chat([sysmsg, ask_python], tools=tools, timeout=min(75.0, left()), template_kwargs=extra_kw)
        okp = "error" not in r and not getattr(agent_cl, "budget_dropped", False)
        out["agent_params"].append({"ok": okp, "effort": (extra_kw or cfg.get("template_kwargs") or {}),
                                    "sec": round(time.time() - t, 1), "error": (r.get("error") or "")[:300],
                                    "finish": r.get("finish_reason")})
    out["agent_params_ok"] = bool(out["agent_params"]) and all(x["ok"] for x in out["agent_params"])
    # v1.6: the DIRECT rescue's two request shapes (its effort + 32k budget; the no-thinking commit call), 32 output
    # tokens each. Informational: a rejection is reported, it does not fail the smoke (the agent can still run)
    out["rescue_params"] = []
    if cfg.get("rescue"):
        from rescue import direct_client
        rcl = direct_client(cfg)
        for extra_kw in (None, {"enable_thinking": False, "reasoning_effort": "low"}):
            if left() <= 5:
                out["deadline_hit"] = True
                break
            t = time.time()
            r = rcl.chat([sysmsg, {"role": "user", "content": "Reply with the digit 7."}], max_tokens=32,
                         timeout=min(75.0, left()), template_kwargs=extra_kw)
            out["rescue_params"].append({"ok": "error" not in r and not getattr(rcl, "budget_dropped", False),
                                         "commit_call": extra_kw is not None, "sec": round(time.time() - t, 1),
                                         "error": (r.get("error") or "")[:300], "finish": r.get("finish_reason")})
    out["rescue_params_ok"] = (all(x["ok"] for x in out["rescue_params"]) if out["rescue_params"] else None)
    # roundtrip on the mode the agent will actually use (thinking if it passed, else plain)
    mode = "thinking" if out["thinking_ok"] and "thinking" in first_python else "auto"
    out["roundtrip_mode"] = mode
    if mode in first_python:
        cl, msgs, r1, tool_out = first_python[mode]
        c = r1["tool_calls"][0]
        follow = msgs + [
            {"role": "assistant", "content": r1.get("content") or "",
             "tool_calls": [{"id": c["id"], "type": "function",
                             "function": {"name": "python", "arguments": json.dumps(c["arguments"])}}]},
            {"role": "tool", "tool_call_id": c["id"], "content": tool_out},
            {"role": "user", "content": "Good. Now call the `submit` tool with a Python program that defines "
                                        "transform(grid) and returns grid unchanged."}]
        for _ in range(2):
            t = time.time()
            r2 = ask(cl, follow, None)
            if r2 is None:
                break
            ok, why, _ = check(r2, "submit")
            out["roundtrip"].append(rec(r2, ok, why, t))
            if ok:
                break
    out["roundtrip_ok"] = any(x["ok"] for x in out["roundtrip"])
    out["ok"] = (out["auto_ok"] or out["thinking_ok"]) and out["roundtrip_ok"] and out["agent_params_ok"]
    print("tool smoke:", {k: out[k] for k in ("forced_ok", "auto_ok", "thinking_ok", "thinking_budget_retained",
                                              "roundtrip_ok", "roundtrip_mode", "agent_params_ok",
                                              "rescue_params_ok", "deadline_hit", "ok")}, flush=True)
    return out
WORK = Path("/kaggle/working")
SITE = WORK / "vllm-site-packages"
LOG = WORK / "vllm-server.log"


def one(pattern):
    hits = glob.glob(pattern, recursive=True)
    if not hits:
        raise FileNotFoundError(pattern)
    return Path(hits[0])


def install(cfg):
    wh = one("/kaggle/input/**/arc3-vllm-h100-wheelhouse-v3/**/requirements.lock").parent
    t = time.time()
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(wh),
                    "--requirement", str(wh / "requirements.lock"), "--target", str(SITE),
                    "--upgrade", "--ignore-installed", "--only-binary", ":all:", "--no-compile",
                    "--disable-pip-version-check", "--no-warn-conflicts", "-q"], check=True,
                   timeout=float(cfg.get("install_timeout_s", 1800)))   # v1.4
    print(f"wheelhouse installed in {time.time() - t:.0f}s", flush=True)
    if cfg.get("overlay"):
        wheels = sorted(glob.glob(cfg["overlay"], recursive=True))
        if not wheels:
            raise FileNotFoundError(cfg["overlay"])
        for pat in ("transformers", "transformers-*.dist-info", "huggingface_hub",
                    "huggingface_hub-*.dist-info"):
            for p in SITE.glob(pat):
                shutil.rmtree(p) if p.is_dir() else p.unlink()
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                        "--target", str(SITE), "--upgrade", "--force-reinstall", "--no-compile",
                        "--disable-pip-version-check", "-q", *wheels], check=True, timeout=600)   # v1.4
        print("overlay installed:", [Path(w).name for w in wheels], flush=True)


def agent_e2e(cfg, e2e_deadline):
    """v1.7 (GPT final audit, Patch B): the agent's REAL request -- its effort, sampling, thinking budget and full
    per-turn max_tokens, with the agent's tools -- on a fresh random task: a python call whose code really runs, the
    output sent back as a role "tool" message, then a submit that verifies. Tried with the configured effort first,
    then each effort-ladder step; returns the first effort that passes (ok False if none does). Each request is bounded
    (<= 300 s) and the whole probe by e2e_deadline."""
    sys.path.insert(0, HERE)
    from client import ChatClient
    from repl_agent import make_tools, verify
    from repl import run_snippet
    tools = make_tools(cfg.get("retain_functions", False))
    base_tk = dict(cfg.get("template_kwargs") or {})
    efforts = [base_tk.get("reasoning_effort")] + [e for e in (cfg.get("effort_ladder") or [])
                                                   if e != base_tk.get("reasoning_effort")]
    sysmsg = {"role": "system", "content": "You solve grid puzzles with the tools. Answer only with the requested "
                                           "tool call."}
    ask_py = {"role": "user", "content": "Call the `python` tool with code that prints train_inputs[0].shape."}
    out = {"ok": False, "effort": None, "tries": []}

    def first_call(r, name):
        calls = r.get("tool_calls") or []
        if "error" in r or not calls or calls[0].get("name") != name or not isinstance(calls[0].get("arguments"), dict):
            return None, None
        code = calls[0]["arguments"].get("code")
        return (calls[0], code) if isinstance(code, str) and code.strip() else (None, None)

    for eff in efforts:
        left = e2e_deadline - time.time()
        if left < 60:
            out["deadline_hit"] = True
            break
        task, shape = _smoke_task()
        cl = ChatClient("http://127.0.0.1:1234/v1", "qwen", temperature=cfg.get("temperature", 0.6),
                        top_p=cfg.get("top_p", 0.95), top_k=cfg.get("top_k", 20),
                        max_tokens=int(cfg.get("turn_max_tokens") or cfg.get("max_tokens") or 8192),
                        thinking=cfg.get("thinking", True), thinking_token_budget=cfg.get("thinking_token_budget"),
                        template_kwargs=dict(base_tk, **({"reasoning_effort": eff} if eff else {})))
        t = time.time()
        r1 = cl.chat([sysmsg, ask_py], tools=tools, timeout=min(300.0, left))
        rec = {"effort": eff, "python_s": round(time.time() - t, 1), "python_finish": r1.get("finish_reason"),
               "python_tokens": r1.get("completion_tokens"), "error": (r1.get("error") or "")[:300], "ok": False}
        c1, code1 = first_call(r1, "python")
        o = run_snippet(code1, task, timeout=8.0) if code1 else ""
        rec["python_ok"] = bool(code1) and shape in o and "Traceback" not in o and "[snippet crashed]" not in o
        if rec["python_ok"] and e2e_deadline - time.time() > 30:
            follow = [sysmsg, ask_py,
                      {"role": "assistant", "content": r1.get("content") or "",
                       "tool_calls": [{"id": c1["id"], "type": "function",
                                       "function": {"name": "python", "arguments": json.dumps(c1["arguments"])}}]},
                      {"role": "tool", "tool_call_id": c1["id"], "content": o},
                      {"role": "user", "content": "Good. Now call the `submit` tool with a Python program that "
                                                  "defines transform(grid) and returns grid unchanged."}]
            t = time.time()
            r2 = cl.chat(follow, tools=tools, timeout=min(300.0, e2e_deadline - time.time()))
            rec.update(submit_s=round(time.time() - t, 1), submit_finish=r2.get("finish_reason"),
                       submit_tokens=r2.get("completion_tokens"), error2=(r2.get("error") or "")[:300])
            _, code2 = first_call(r2, "submit")
            if code2:
                n_ok, test_out, _fb = verify(code2, task, timeout=8.0)
                rec["ok"] = n_ok == 1 and test_out == [task["test"][0]["input"]]
        rec["budget_dropped"] = bool(getattr(cl, "budget_dropped", False))
        rec["ok"] = rec["ok"] and not rec["budget_dropped"]
        out["tries"].append(rec)
        print("agent e2e:", rec, flush=True)
        if rec["ok"]:
            out.update(ok=True, effort=eff)
            break
    return out


def start_server(cfg, extra, timeout=1500):
    model = one(cfg["model_glob"]).parent
    env = dict(os.environ, PYTHONPATH=str(SITE), USE_TF="0", TRANSFORMERS_NO_TF="1",
               TRANSFORMERS_NO_TORCHVISION="1", VLLM_NO_USAGE_STATS="1",
               LIBRARY_PATH=f"{LINK_DIR}:/usr/local/nvidia/lib64:" + os.environ.get("LIBRARY_PATH", ""),
               VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS="1")   # v1.1: libcuda shim dir first; graph memory in budget
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", str(model),
           "--served-model-name", "qwen", "--host", "127.0.0.1", "--port", "1234",
           "--tensor-parallel-size", "4", "--enable-auto-tool-choice", *cfg["parsers"],
           "--generation-config", "vllm", "--enable-prefix-caching",
           "--max-model-len", str(cfg.get("max_model_len", 65536)), "--gpu-memory-utilization", "0.90",
           "--max-num-seqs", str(cfg.get("max_num_seqs", 32)),
           # v1.1: 4096 = the batch step that fixed Q27-XHIGH run 2's MTP-drafter OOM (default prefill step was larger)
           "--max-num-batched-tokens", str(cfg.get("max_num_batched_tokens", 4096)), *extra]
    print(" ".join(cmd), flush=True)
    t = time.time()
    # "w": a retry must not see the previous attempt's "EngineCore failed".
    proc = subprocess.Popen(cmd, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
                            start_new_session=True)
    CURRENT["proc"] = proc
    SERVER_PID.write_text(str(proc.pid))                 # v1.1: the notebook kills this group if we are killed
    while time.time() - t < timeout:
        text = LOG.read_text(errors="replace") if LOG.exists() else ""
        if proc.poll() is not None or "EngineCore failed" in text:
            break
        try:
            urllib.request.urlopen("http://127.0.0.1:1234/v1/models", timeout=5)
            print(f"server ready after {time.time() - t:.0f}s", flush=True)
            return proc
        except Exception:
            time.sleep(10)
    print("server FAILED to start; tail:\n" + "\n".join(
        LOG.read_text(errors="replace").splitlines()[-30:]), flush=True)
    kill(proc)
    return None


def kill(proc):
    """Kill the server's whole process group -- engine workers hold GPU memory -- and wait
    for the GPUs to drain, so a restart does not hit out-of-memory."""
    import signal
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except Exception:
        proc.kill()
    try:
        proc.wait(timeout=60)
    except Exception:
        pass
    t_end = time.time() + 240                       # v1.4: the whole drain wait is bounded
    while time.time() < t_end:
        try:
            used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                  capture_output=True, text=True, timeout=20).stdout.split()
        except Exception:
            used = []
        if used and all(int(u) < 1500 for u in used if u.strip().isdigit()):
            return
        time.sleep(5)


def server_ladder(cfg):
    """Option sets to try in order. An explicit cfg["server_ladder"] wins (plain serving is
    appended if missing); otherwise server_extra + server_try -> server_extra -> plain."""
    if cfg.get("server_ladder"):
        ladder = [list(x) for x in cfg["server_ladder"]]
        return ladder if [] in ladder else ladder + [[]]
    base = cfg.get("server_extra", [])
    ladder = [base + cfg["server_try"]] if cfg.get("server_try") else []
    return ladder + [base] + ([[]] if base else [])


def main(cfg):
    proc = None
    try:
        import signal as _signal
        _signal.signal(_signal.SIGTERM, _on_sigterm)
        t_install = time.time()
        install(cfg)
        install_s = round(time.time() - t_install)
        try:                                 # v1.7: informational -- can a rootless namespace sandbox work here?
            _ns = subprocess.run(["unshare", "--user", "--map-root-user", "--mount", "--fork", "true"],
                                 capture_output=True, text=True, timeout=10)
            KERNEL_ISO["unshare_rc"], KERNEL_ISO["stderr"] = _ns.returncode, _ns.stderr[-200:]
        except Exception as _e:
            KERNEL_ISO["unshare_rc"], KERNEL_ISO["stderr"] = None, repr(_e)[:200]
        link = ensure_libcuda()
        if link.get("probe") == "FAIL":
            # v1.2: logged loudly, not fatal -- only FlashInfer's JIT needs -lcuda; other attention backends on the
            # ladder load without it, and every rung must still pass the tool smoke before the agent starts
            print("WARNING: -lcuda link probe FAILED; rungs that need FlashInfer JIT will fail", flush=True)
        # A ladder of server option sets, most ambitious first; the last is plain serving.
        # Untested-on-L4 options (MTP, --reasoning-config) must never cost the whole run.
        # v1.1: each rung must also pass a real tool-calling smoke test, and the ladder never eats the time the
        # agent needs (min_agent_s before the deadline).
        ladder = server_ladder(cfg)
        report = {"cuda_link": CUDA_LINK, "install_s": install_s, "rungs": [], "kernel_isolation": KERNEL_ISO}
        reserve = float(cfg.get("min_agent_s", 1800))
        for i, extra in enumerate(ladder):
            left = cfg["deadline_epoch"] - time.time() - reserve
            if left < 300:
                print(f"no time for server option set {i} ({left:.0f}s before the agent reserve) -> stop", flush=True)
                break
            t_rung = time.time()
            proc = start_server(cfg, extra, timeout=min(1500.0, left))
            rung = {"index": i, "extra": extra, "server_up": proc is not None, "start_s": round(time.time() - t_rung)}
            if proc is not None:
                # v1.2: the smoke has its own hard deadline inside the time left before the agent reserve
                slack = cfg["deadline_epoch"] - time.time() - reserve
                if slack <= 60:
                    print(f"option set {i}: server up but no time left for the tool smoke -> stop", flush=True)
                    rung["note"] = "no time for the smoke"
                    report["rungs"].append(rung)
                    kill(proc)
                    proc = None
                    break
                t_smoke = time.time()
                sm = smoke(cfg, time.time() + min(240.0, slack - 30.0))   # v1.3: 240 s (adds the roundtrip)
                rung["smoke_s"] = round(time.time() - t_smoke)
                rung["smoke"] = sm
                if sm["ok"]:
                    if not sm["forced_ok"]:
                        cfg["force_submit_after"] = 0      # named tool_choice does not work on this rung
                    if not sm["thinking_ok"]:
                        cfg["thinking"] = False            # explicit fallback: no thinking, so no unbounded thinking
                        cfg["thinking_token_budget"] = None
                    report["rungs"].append(rung)
                    report["chosen"] = {"index": i, "extra": extra, "force_submit_after": cfg.get("force_submit_after"),
                                        "thinking": cfg.get("thinking", True),
                                        "thinking_budget_retained": sm.get("thinking_budget_retained", False)}
                    report["remaining_agent_s"] = round(cfg["deadline_epoch"] - time.time())
                    REPORT_PATH.write_text(json.dumps(report, indent=1, default=str))
                    print(f"server up with option set {i}: {extra}", flush=True)
                    break
                kill(proc)
                proc = None
                if sm.get("roundtrip_ok") and not sm.get("agent_params_ok"):
                    # v1.5: the server works, the AGENT's request is rejected (a config error, the same on every rung)
                    rung["failure"] = "AGENT_PARAMS"
                    report["rungs"].append(rung)
                    REPORT_PATH.write_text(json.dumps(report, indent=1, default=str))
                    print(f"option set {i}: agent request parameters rejected -> stop the ladder", flush=True)
                    break
                print(f"option set {i}: server up but tool smoke FAILED -> next", flush=True)
            else:
                # v1.3: classify the startup failure; no rung changes the attention backend, so a CUDA-linker failure
                # would repeat on every rung -> stop instead of reloading the 27B model again and again
                rung["failure"] = classify_startup_failure()
                if rung["failure"] == "CUDA_LINKER":
                    report["rungs"].append(rung)
                    REPORT_PATH.write_text(json.dumps(report, indent=1, default=str))
                    print(f"option set {i}: CUDA linker failure ({CUDA_LINK.get('probe')}) -> stop the ladder",
                          flush=True)
                    break
            report["rungs"].append(rung)
            REPORT_PATH.write_text(json.dumps(report, indent=1, default=str))
            print(f"option set {i} failed ({rung.get('failure', 'smoke')}) -> next", flush=True)
            time.sleep(20)
        REPORT_PATH.write_text(json.dumps(report, indent=1, default=str))
        if proc is None:
            return
        # v1.7 (GPT final audit): a rejected DIRECT-rescue request shape disables the rescue (the agent keeps the
        # whole time; the eval report marks the three-way comparison incomplete) instead of failing task by task
        chosen_sm = (report["rungs"][-1].get("smoke") or {}) if report["rungs"] else {}
        cfg["rescue_smoke_ok"] = chosen_sm.get("rescue_params_ok")
        if cfg.get("rescue") and chosen_sm.get("rescue_params_ok") is False:
            cfg["rescue"] = False
            report["rescue_disabled"] = "rescue request shape rejected by the server"
            print("WARNING: DIRECT rescue requests rejected -> rescue disabled", flush=True)
        # v1.7 (GPT Patch B): the agent's REAL request end to end before any task (bounded, never fatal)
        e2e_budget = min(float(cfg.get("e2e_max_s", 600)), cfg["deadline_epoch"] - time.time() - 1800)
        if e2e_budget >= 120:
            e2e = agent_e2e(cfg, time.time() + e2e_budget)
            report["agent_e2e"] = e2e
            cfg["agent_e2e_ok"] = e2e["ok"]
            if e2e["ok"] and e2e["effort"] and e2e["effort"] != (cfg.get("template_kwargs") or {}).get("reasoning_effort"):
                lad = list(cfg.get("effort_ladder") or [])
                cfg["template_kwargs"] = dict(cfg.get("template_kwargs") or {}, reasoning_effort=e2e["effort"])
                cfg["effort_ladder"] = lad[lad.index(e2e["effort"]) + 1:] if e2e["effort"] in lad else lad
                print(f"agent effort switched to {e2e['effort']} (the configured effort failed end to end)", flush=True)
            elif not e2e["ok"]:
                print("WARNING: agent end-to-end probe failed on every effort; running with the configured effort",
                      flush=True)
        else:
            report["agent_e2e"] = {"ok": None, "skipped": f"only {e2e_budget:.0f}s available"}
        report["agent_config"] = {k: cfg.get(k) for k in ("template_kwargs", "effort_ladder", "thinking",
                                                          "thinking_token_budget", "rescue", "rescue_smoke_ok")}
        REPORT_PATH.write_text(json.dumps(report, indent=1, default=str))
        sys.path.insert(0, HERE)
        if cfg.get("eval"):                  # Q27-AGENT: commit run = evaluation subset, scored
            import run_eval
            run_eval.main(dict(cfg, base_url="http://127.0.0.1:1234/v1", model="qwen"))
        else:
            import run_submit
            run_submit.main(dict(cfg, base_url="http://127.0.0.1:1234/v1", model="qwen"))
    except BaseException:
        print("submit_main caught:\n" + traceback.format_exc(), flush=True)
    finally:
        if proc is not None:
            kill(proc)
        shutil.rmtree(SITE, ignore_errors=True)


if __name__ == "__main__":
    try:
        main(json.loads(sys.argv[1]))
    except BaseException:
        print(traceback.format_exc(), flush=True)
    sys.exit(0)


In [ ]:
%%writefile /kaggle/working/agent/rescue.py
"""HYBRID rescue: the LB-18.75 DIRECT method for every test input whose task the agent did not verify.

v1.8 (run-record check, 2026-10-12): v1.6/v1.7 asked one 32k-token answer per TASK with another prompt. The EVAL12
records of our 18.75 pipeline (runs 4 + 7) show every correct answer needed 26-37k thinking tokens (one more came from
a forced close at 49k), so 32k would have lost 3 of its 5 hits. Now exactly as Q27-XHIGH v1.4 (LB 18.75): one request
per TEST INPUT, its prompt and grid parser copied verbatim, xhigh, temperature 0.6 / top_p 0.95 / top_k 20, thinking
capped at 49,152 tokens and up to 8,192 answer tokens. Difference: the server forces </think> at the budget (v1.4
appended Qwen's stop sentence); 1 of 37 forced closes was correct there, the natural closes carry the method.
A reply without a parseable grid gets ONE no-thinking commit call with the tail of its notes.

Candidate order per test input (merge): the top verified program grid -> the direct grid -> further verified grids ->
unverified grids -> the test input; the first two distinct grids are the attempts. v1.8: only ONE verified grid ranks
above the direct answer (R6 code loop: 3 of 4 train-verified programs were wrong on test).
"""
import json
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor

from client import ChatClient


# ---------------------------------------------------------------- verbatim from Q27-XHIGH v1.4 (LB 18.75), cells 4-5
def render_grid(g):
    return "\n".join(" ".join(str(int(v)) for v in row) for row in g)


def build_user_message(train, test_input):
    parts = ["You are solving an abstract reasoning puzzle. Each example has an "
             "INPUT grid and its OUTPUT grid (digits 0-9 are colors; 0 is usually "
             "background). Infer the ONE rule that maps every input to its output, "
             "then apply it to the FINAL INPUT.\n"]
    for i, p in enumerate(train):
        parts.append("Example %d INPUT:\n%s\nExample %d OUTPUT:\n%s\n" %
                     (i + 1, render_grid(p["input"]), i + 1, render_grid(p["output"])))
    parts.append("FINAL INPUT:\n%s\n" % render_grid(test_input))
    parts.append("Reason step by step however you like. Then give ONLY the final "
                 "output grid as rows of space-separated digits inside a fenced "
                 "block:\n```grid\n<rows>\n```\nOutput nothing after the block.")
    return "\n".join(parts)


def valid_grid(g):
    return (isinstance(g, list) and 1 <= len(g) <= 30 and all(isinstance(r, list) and 1 <= len(r) <= 30 for r in g)
            and len({len(r) for r in g}) == 1
            and all(isinstance(v, int) and not isinstance(v, bool) and 0 <= v <= 9 for r in g for v in r))


_FENCE_RE = re.compile(r"```([^\n`]*)\n(.*?)```", re.S)
_SEP_ROW = re.compile(r"\[?\s*\d(?:\s*[, ]\s*\d)*\s*\]?,?")
_JSON_ROW = re.compile(r"\[\s*\d(?:\s*,\s*\d)*\s*\]")


def _parse_row(s):
    s = s.strip()
    if not s:
        return None
    if re.fullmatch(r"\d+", s):                                  # contiguous digits, one per cell ("102")
        return [int(c) for c in s]
    if _SEP_ROW.fullmatch(s):
        return [int(c) for c in re.findall(r"\d", s)]
    return None


def _first_row_run(lines):
    rows = []
    for line in lines:
        r = _parse_row(line)
        if r is None:
            if rows:
                break
            continue
        rows.append(r)
    return rows if valid_grid(rows) else None


def _grid_from_obj(obj):
    if isinstance(obj, dict):
        for k in ("output", "grid", "test_output", "answer", "test_outputs"):
            if k in obj:
                return _grid_from_obj(obj[k])
        return None
    if valid_grid(obj):
        return obj
    if isinstance(obj, list) and obj and valid_grid(obj[-1]):     # a list of grids: the last one
        return obj[-1]
    return None


def _json_rows_grid(b):
    grids, cur = [], []
    for m in _JSON_ROW.finditer(b):
        cur.append(json.loads(m.group(0)))
        if b[m.end():].lstrip().startswith("]"):
            grids.append(cur); cur = []
    if cur:
        grids.append(cur)
    for g in reversed(grids):
        if valid_grid(g):
            return g
    return None


def _grid_from_block(body):
    b = body.strip()
    if b[:1] in ("[", "{"):
        try:
            g = _grid_from_obj(json.loads(b))
        except Exception:
            g = None
        if g is None:
            g = _json_rows_grid(b)
        if g is not None:
            return g
    return _first_row_run(b.splitlines())


def parse_grid(text):
    """answer text after </think> -> grid or None: the last fenced block (any label) that holds a valid grid; else the
    last run of digit rows in the text (unfenced answer or a fence that was never closed)"""
    for _lab, body in reversed(_FENCE_RE.findall(text or "")):
        g = _grid_from_block(body)
        if g is not None:
            return g
    runs, cur = [], []
    for line in (text or "").splitlines():
        r = _parse_row(line)
        if r is None:
            if cur:
                runs.append(cur); cur = []
        else:
            cur.append(r)
    if cur:
        runs.append(cur)
    for run in reversed(runs):
        if valid_grid(run):
            return run
    return None
# ---------------------------------------------------------------- end of the verbatim block


COMMIT = """{prompt}

## Your notes so far (they were cut off)
{notes}

Stop analysing now. Apply the rule you were converging on and give ONLY the final output grid as rows of space-separated digits inside a fenced block:
```grid
<rows>
```"""


def direct_client(cfg):
    """v1.8: base_url / model default to the local server (v1.7 crashed: the smoke passed a cfg without base_url)"""
    think = int(cfg.get("rescue_think", 49152))
    use_think = bool(cfg.get("thinking", True))
    return ChatClient(cfg.get("base_url", "http://127.0.0.1:1234/v1"), cfg.get("model", "qwen"),
                      max_tokens=think + int(cfg.get("rescue_answer", 8192)),
                      temperature=cfg.get("rescue_temperature", 0.6), top_p=0.95, top_k=20, thinking=use_think,
                      thinking_token_budget=(think if use_think and cfg.get("thinking_token_budget") else None),
                      template_kwargs={"reasoning_effort": cfg.get("rescue_effort", "xhigh")})


def direct_answer(client, task, j, seed, deadline, commit_tokens=8192, notes_chars=16000):
    """one LB-18.75 request for test input j; returns a record with "grid" (or None)"""
    prompt = build_user_message(task["train"], task["test"][j]["input"])

    def timeout():
        return max(60.0, min(4 * 3600, deadline - time.time() + 60))

    mt = client.max_tokens
    for _ in range(3):
        r = client.chat([{"role": "user", "content": prompt}], seed=seed, max_tokens=mt, timeout=timeout())
        if not (r.get("context_overflow") and mt > 16384):
            break
        mt = int(mt * 0.8)                     # a very long prompt: shrink the reply, never give up
    if "error" in r:
        return {"status": "error: " + str(r["error"])[:200], "grid": None, "gen_tokens": 0}
    rec = {"finish": r.get("finish_reason"), "gen_tokens": int(r.get("completion_tokens") or 0), "max_tokens": mt,
           "reasoning_chars": len(r.get("reasoning") or ""), "answer_tail": (r.get("content") or "")[-1200:],
           "reasoning_tail": (r.get("reasoning") or "")[-2000:]}
    grid = parse_grid(r.get("content") or "")
    if grid is not None:
        return dict(rec, status="answered", grid=grid)
    notes = ((r.get("reasoning") or "") + "\n" + (r.get("content") or "")).strip()
    if notes and deadline - time.time() > 120:
        r2 = client.chat([{"role": "user", "content": COMMIT.format(prompt=prompt, notes=notes[-notes_chars:])}],
                         seed=seed + 1, max_tokens=commit_tokens, timeout=timeout(),
                         template_kwargs={"enable_thinking": False, "reasoning_effort": "low"})
        if "error" in r2:
            rec["commit_error"] = str(r2["error"])[:200]
        else:
            rec["gen_tokens"] += int(r2.get("completion_tokens") or 0)
            rec["commit_finish"] = r2.get("finish_reason")
            grid = parse_grid(r2.get("content") or "") or parse_grid(r2.get("reasoning") or "")
            if grid is not None:
                return dict(rec, status="committed", grid=grid)
    return dict(rec, status="no grid", grid=None)


def run_rescue(task_ids, tasks, cfg, deadline, log=print, on_result=None):
    """direct answers for every test input of task_ids (in that order), concurrently.
    Returns ({tid: {"grids": [grid or None per test input], "units": {j: rec}, "gen_tokens", "status"}}, client)."""
    client = direct_client(cfg)
    out, lock = {}, threading.Lock()
    for tid in task_ids:
        out[tid] = {"grids": [None] * len(tasks[tid]["test"]), "units": {}, "gen_tokens": 0, "status": "pending"}
    units = [(tid, j) for tid in task_ids for j in range(len(tasks[tid]["test"]))]

    def work(unit):
        tid, j = unit
        if deadline - time.time() < float(cfg.get("rescue_min_start_s", 300)):
            rec = {"status": "skipped: deadline", "grid": None, "gen_tokens": 0}
        else:
            t = time.time()
            try:
                rec = direct_answer(client, tasks[tid], j, 7 + 1000 * j, deadline,
                                    commit_tokens=int(cfg.get("rescue_commit_tokens", 8192)))
            except Exception as e:                       # one unit must never sink the stage
                rec = {"status": f"crashed: {type(e).__name__}: {e}"[:200], "grid": None, "gen_tokens": 0}
            rec["wall"] = round(time.time() - t, 1)
        with lock:
            tr = out[tid]
            tr["units"][j] = rec
            tr["grids"][j] = rec.get("grid")
            tr["gen_tokens"] += int(rec.get("gen_tokens") or 0)
            sts = [str(u.get("status")).split(":")[0] for u in tr["units"].values()]
            tr["status"] = sts[0] if len(set(sts)) == 1 else "mixed"
            if on_result:
                try:
                    on_result(tid, tr)
                except Exception as e:
                    log(f"[rescue] on_result failed for {tid}: {e}")
        log(f"[rescue] {tid}#{j} {str(rec.get('status'))[:30]} tok {rec.get('gen_tokens', 0)} "
            f"finish {rec.get('finish')}")

    with ThreadPoolExecutor(max_workers=int(cfg.get("rescue_concurrency", cfg.get("concurrency", 16)))) as ex:
        list(ex.map(work, units))
    return out, client


def direct_grid(rec, j):
    grids = (rec or {}).get("grids")
    return grids[j] if isinstance(grids, list) and j < len(grids) else None


def merge(agent_ranked, direct, inp):
    """agent_ranked: [(grid, source)] best first, source 'verified ...' or 'unverified ...'; direct: grid or None"""
    ver = [g for g, s in agent_ranked if str(s).startswith("verified")]
    unv = [g for g, s in agent_ranked if not str(s).startswith("verified")]
    order = (ver[:1] + [direct] + ver[1:] + unv) if direct is not None else (ver + unv)
    out = []
    for g in order + [inp]:
        if g is not None and g not in out:
            out.append(g)
        if len(out) == 2:
            break
    while len(out) < 2:
        out.append(inp)
    return {"attempt_1": out[0], "attempt_2": out[1]}


def hybrid_submission(answers, rescue_recs, tasks):
    return {t: [merge(answers.get(t, [[]] * len(task["test"]))[j], direct_grid(rescue_recs.get(t), j), x["input"])
                for j, x in enumerate(task["test"])] for t, task in tasks.items()}


def direct_submission(rescue_recs, tasks):
    sub = {}
    for t, task in tasks.items():
        rows = []
        for j, x in enumerate(task["test"]):
            g = direct_grid(rescue_recs.get(t), j)
            rows.append({"attempt_1": g if g is not None else x["input"], "attempt_2": x["input"]})
        sub[t] = rows
    return sub


def hits(sub, sol, ids):
    """per-query correctness (Kaggle metric: either attempt exact)"""
    return {(t, j): int(s in (sub[t][j]["attempt_1"], sub[t][j]["attempt_2"]))
            for t in ids for j, s in enumerate(sol[t])}


def load_direct_ref(path, run="run7"):
    """v1.8: the LB-18.75 pipeline's own answers on EVAL12 (runs 4 and 7, parsed with the verbatim parser) in the
    rescue-record shape, so the commit run can merge the agent with DIRECT exactly without spending GPU on DIRECT"""
    ref = json.load(open(path))
    return {t: {"grids": [q.get("grid") for q in qs], "gen_tokens": sum(int(q.get("think_tokens") or 0) for q in qs),
                "status": "reference", "units": {j: q for j, q in enumerate(qs)}}
            for t, qs in ref[run].items()}


In [ ]:
%%writefile /kaggle/working/agent/direct_ref_eval12.json
{"run7":{"136b0064":[{"grid":[[0,0,5,0,0,0,0],[0,0,1,1,1,0,0],[0,0,0,2,2,0,0],[0,0,0,6,0,0,0],[0,0,0,6,0,0,0],[3,3,3,3,0,0,0],[6,0,0,0,0,0,0],[6,0,0,0,0,0,0],[1,1,1,0,0,0,0],[0,0,1,1,1,0,0],[0,0,0,0,6,0,0],[0,0,0,0,6,0,0],[0,3,3,3,3,0,0],[2,2,0,0,0,0,0],[0,0,0,0,0,0,0],[0,0,0,0,0,0,0],[0,0,0,0,0,0,0],[0,0,0,0,0,0,0],[0,0,0,0,0,0,0]],"think_tokens":25934,"end_reason":"natural","correct":1}],"20a9e565":[{"grid":[[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3],[3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3],[3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3],[3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3],[3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3],[3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3],[3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3],[3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"21897d95":[{"grid":[[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,4,4,4,4,4,4,4,4,4,4,4,4,6,6,6,6,6,6],[2,2,2,2,2,2,4,4,4,4,4,4,4,4,4,4,4,4,6,6,6,6,6,6],[2,2,2,2,2,2,4,4,4,4,4,4,4,4,4,4,4,4,6,6,6,6,6,6],[2,2,2,2,2,2,4,4,4,4,4,4,4,4,4,4,4,4,6,6,6,6,6,6],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"247ef758":[{"grid":[[0,0,0,0,3,9,9,1,9,9,3,9,9,9,4,6,9,9,8,9],[0,0,0,0,3,9,0,0,0,0,0,0,0,0,0,0,0,0,0,9],[0,0,0,0,3,9,0,1,0,0,0,0,0,0,0,0,0,0,0,9],[0,0,0,0,3,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1],[0,0,0,0,3,9,0,1,0,0,0,0,0,0,0,0,0,0,0,9],[0,0,0,0,3,9,0,0,0,0,0,0,0,0,0,0,0,0,0,9],[0,0,0,0,3,3,0,0,0,0,0,0,0,4,0,0,0,0,0,3],[0,0,0,0,3,4,0,0,0,0,0,0,0,4,4,0,0,0,0,4],[0,0,0,0,3,4,0,0,0,0,0,0,0,0,4,4,6,0,0,4],[0,0,0,0,3,6,0,0,0,0,0,0,0,0,6,4,6,0,0,6],[0,0,0,0,3,9,0,0,0,0,0,0,0,0,6,0,6,0,0,9],[0,0,0,0,3,9,0,1,0,0,0,0,0,0,0,0,0,0,0,9],[0,7,0,7,3,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1],[0,0,7,0,3,9,0,1,0,0,0,0,0,0,0,0,0,0,0,9],[0,7,0,7,3,9,8,1,9,9,3,9,9,9,4,6,9,9,9,9]],"think_tokens":36770,"end_reason":"natural","correct":1},{"grid":[[0,0,0,0,0,2,9,1,7,1,7,3,1,4,6,1,1,1],[0,0,0,0,0,2,1,0,7,0,7,0,0,0,0,0,0,1],[0,0,0,0,0,2,7,7,7,7,7,7,0,0,0,0,0,7],[0,0,0,0,0,2,1,0,7,0,7,0,0,0,0,0,0,1],[8,8,8,0,0,2,3,0,0,0,0,0,0,4,4,0,0,3],[0,0,0,0,0,2,4,0,0,0,0,0,0,4,0,0,0,4],[0,0,0,0,0,2,1,0,0,0,0,0,4,4,0,0,0,1],[0,0,0,0,0,2,1,0,0,0,0,0,0,6,0,0,0,1],[0,0,0,0,0,2,6,0,0,0,0,0,0,0,6,0,0,6],[0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,6,0,1],[0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,6,1],[0,0,0,0,0,2,1,1,7,1,7,3,1,4,6,1,1,1]],"think_tokens":46470,"end_reason":"natural","correct":0}],"2b83f449":[{"grid":[[8,8,8,8,8,8,8,8,8,8,6,8,8,8,8,8,8],[0,0,0,0,0,0,0,0,0,8,6,8,0,0,0,0,0],[3,3,8,8,8,8,8,8,8,8,6,8,8,8,8,8,8],[0,0,0,0,0,0,0,0,0,0,0,0,8,6,8,0,0],[8,8,8,8,8,8,8,0,3,8,8,8,8,6,8,8,8],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[8,8,8,8,8,8,8,6,8,8,8,8,8,8,8,8,8],[0,0,0,0,0,0,8,6,8,0,0,0,0,0,0,0,0],[3,3,8,8,8,8,8,6,8,8,8,8,8,8,8,8,8],[0,0,8,6,8,0,0,0,0,0,0,0,0,0,0,0,0],[3,8,8,6,8,8,8,8,8,8,0,8,8,8,8,8,3],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[8,8,8,8,8,8,8,8,8,8,6,8,8,8,8,8,8],[0,0,0,0,0,0,0,0,0,8,6,8,0,0,0,0,0],[3,3,8,8,8,8,8,8,8,8,6,8,8,8,8,8,8],[0,0,0,0,0,0,8,6,8,0,0,0,0,0,0,0,0],[8,8,8,8,8,8,8,6,8,8,8,8,8,8,8,8,8],[0,0,8,6,8,0,0,0,0,0,0,8,6,8,0,0,0],[8,8,8,8,8,8,8,0,3,8,8,8,8,6,8,8,8]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"4c3d4a41":[{"grid":[[0,0,0,0,0,0,0,0,0,5,5,5,5,5,5,5,5,5,5,5],[0,0,0,0,0,0,0,0,0,5,0,5,0,6,0,3,0,8,0,5],[0,0,0,0,0,0,0,0,0,5,0,9,0,5,0,5,0,8,0,5],[0,0,0,0,0,0,0,0,0,5,0,5,0,5,0,5,0,8,0,5],[0,0,0,0,0,0,0,0,0,0,0,5,0,5,0,5,0,5,0,5],[0,0,0,0,0,0,0,0,0,0,0,5,5,5,5,5,5,5,0,5],[0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,5],[0,0,0,0,0,0,0,0,0,5,5,5,5,5,5,5,5,5,5,5]],"think_tokens":49175,"end_reason":"forced_close","correct":1},{"grid":[[0,0,0,0,0,0,0,0,0,5,5,5,5,5,5,5,5,5,5,5],[0,0,0,0,0,0,0,0,0,5,0,3,0,2,0,4,0,7,0,5],[0,0,0,0,0,0,0,0,0,5,0,3,0,2,0,4,0,7,0,5],[0,0,0,0,0,0,0,0,0,5,0,3,0,2,0,4,0,7,0,5],[0,0,0,0,0,0,0,0,0,0,0,5,0,5,0,5,0,5,0,5],[0,0,0,0,0,0,0,0,0,0,0,5,5,5,5,5,5,5,0,5],[0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,5],[0,0,0,0,0,0,0,0,0,5,5,5,5,5,5,5,5,5,5,5]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"4c7dc4dd":[{"grid":[[0,0,0,0,0],[0,0,2,0,0],[0,2,8,2,0],[0,0,2,0,0],[0,0,0,0,0]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[0,0,0,0,0],[8,2,2,2,2],[2,0,0,0,0],[2,0,0,0,0],[2,0,0,0,0]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"6e4f6532":[{"grid":[[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,2,2,7,7,7,7,7,7,7,7,7,7,3,3]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,8,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,8,9,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,8,9,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,8,8,8,0,0],[1,1,0,0,0,0,0,0,0,0,0,8,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,7,7],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,7,7],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"88bcf3b4":[{"grid":[[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,8,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,8,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,8,5,5,5,5,5,9,9,5,5,5,5,5,5,5,5,5,5],[5,5,5,8,5,5,5,5,5,5,9,9,9,5,5,5,5,5,5,5,5,5],[5,5,8,5,5,5,5,5,5,5,9,9,9,9,5,5,5,5,5,5,5,5],[5,8,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[3,3,3,3,3,3,3,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,4,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,4,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,4,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,6,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,4,4,1,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,4,4,4,5,5,1,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,4,5,5,5,5,5,1,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,1,5,5,5,5,5,5,5,5,5,5,5,5,5]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[8,8,8,8,8,8,8,8,8,8,8,8,8,3,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,3,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,3,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,3,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,3,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,3,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,9,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,9,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,9,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,2,2,2,2,2,2],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,5,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,6,6,8,8,8,8,5,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,6,6,8,8,8,5,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,5,5,8,8,8],[8,8,8,8,5,5,5,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,5,5,8,8],[8,8,8,8,5,5,5,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,4,4,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,4,4,4,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,4,8,8,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"898e7135":[{"grid":[[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"8b7bacbf":[{"grid":[[6,1,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,3,8],[1,1,1,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,3,3],[5,5,5,1,9,9,1,1,1,1,1,1,1,9,9,9,9,9,9,9,5,5,5,5,5,5,3,3,5,5],[5,5,5,9,6,6,9,5,5,5,5,5,9,8,8,8,8,8,8,8,9,5,5,5,5,5,3,5,5,5],[5,5,5,9,6,6,9,5,5,5,5,5,9,8,8,8,8,8,8,8,9,5,5,5,5,3,3,5,5,5],[5,5,5,5,9,9,5,5,5,5,5,5,5,9,9,9,9,9,9,9,1,1,1,1,5,3,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,1,3,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,3,1,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,3,5,5,1,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,9,9,3,5,5,1,5,5],[5,5,5,5,9,9,9,9,9,9,9,5,5,5,5,5,9,3,3,5,5,9,8,8,9,5,5,1,5,5],[5,5,5,9,8,8,8,8,8,8,8,9,5,5,5,9,9,9,3,3,3,9,8,8,9,5,5,1,5,5],[5,5,5,9,8,8,8,8,8,8,8,9,5,5,5,9,5,9,5,5,3,3,9,9,5,5,5,1,1,5],[5,5,3,3,9,9,9,9,9,9,9,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,5,5,1,5],[5,0,3,5,5,5,5,5,5,5,5,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,5,5,1,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,5,1,1,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,5,1,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,5,1,5,5],[5,5,0,5,5,5,5,5,5,5,5,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,1,1,5,5],[5,5,3,5,5,5,5,5,5,5,5,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,1,5,5,5],[5,5,3,5,5,5,5,5,5,5,5,5,5,3,3,3,9,5,5,5,5,5,5,5,5,1,1,5,5,5],[5,5,3,3,5,5,5,5,5,5,3,3,3,3,5,5,5,5,5,5,9,9,9,1,1,1,5,5,5,5],[5,5,5,3,9,9,3,3,3,3,3,5,5,5,5,5,5,5,5,9,6,6,9,1,1,5,5,5,5,5],[5,5,5,9,5,5,9,5,5,5,5,5,5,5,5,5,5,5,9,5,6,6,9,5,5,5,5,5,5,5],[5,5,5,9,5,5,9,5,5,5,5,5,5,5,5,5,5,9,5,5,6,6,9,5,5,5,5,5,5,5],[5,5,5,5,9,9,5,5,5,5,5,5,5,5,5,5,9,5,5,5,9,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,9,5,5,5,9,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,9,9,9,9,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,8,8,0],[0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,1,1,0,0,0,0,0,8,4,4,8],[0,0,0,0,2,2,1,1,0,0,0,0,0,0,0,0,1,1,8,8,0,0,8,4,4,8],[0,0,0,2,4,4,2,0,0,0,0,0,0,0,0,0,0,8,4,4,8,0,0,8,8,0],[0,0,0,2,4,4,2,0,0,0,0,0,0,0,0,0,0,8,4,4,8,0,0,0,0,0],[0,0,0,1,2,2,1,0,0,0,0,0,0,0,0,0,0,0,8,8,1,0,0,0,0,0],[0,0,1,1,0,0,1,1,6,6,6,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0],[0,0,1,0,0,0,0,6,4,4,4,6,0,0,0,8,8,0,0,0,0,0,0,0,1,0],[0,0,1,0,0,0,0,6,4,4,4,6,0,0,8,4,4,8,0,0,0,5,5,5,1,0],[0,0,1,0,0,0,0,6,4,4,4,6,0,0,8,4,4,8,0,0,5,4,4,4,5,0],[0,0,1,0,0,0,0,0,6,6,6,0,0,9,9,8,8,9,0,0,5,4,4,4,5,0],[0,0,1,0,0,0,0,0,0,0,0,0,9,0,0,0,0,9,9,0,0,5,5,5,0,0],[0,0,1,0,0,0,0,0,8,8,9,9,0,0,0,0,0,0,9,0,0,0,0,0,0,0],[0,0,1,0,0,0,0,8,4,4,8,0,0,0,0,0,0,0,9,0,0,0,0,0,0,0],[0,0,1,0,0,0,0,8,4,4,8,0,0,0,0,0,0,0,9,0,0,0,0,0,0,0],[0,0,1,0,0,0,0,0,8,8,0,0,0,0,0,0,0,0,9,0,0,0,8,8,0,0],[0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,8,8,9,0,0,8,4,4,8,0],[0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,8,4,4,8,0,0,8,4,4,8,0],[0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,8,4,4,8,0,0,9,8,8,0,0],[0,1,0,0,0,0,0,9,9,9,9,0,0,0,0,9,8,8,9,9,9,0,0,0,0,0],[1,1,0,0,9,9,9,0,0,0,9,9,9,9,9,9,0,0,0,0,0,0,0,0,0,0],[4,1,0,0,9,3,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"b6f77b65":[{"grid":[[3,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,4,0,0,0,0,7,7,7,0,0],[0,0,4,0,0,0,0,0,0,7,0,0],[0,0,4,0,0,0,0,0,0,7,0,0],[0,0,6,5,5,5,5,5,0,7,0,0],[0,0,6,0,0,0,0,5,0,7,0,0],[0,0,6,0,0,0,0,5,0,7,0,0],[0,0,1,1,1,0,2,2,2,2,9,0],[0,0,0,0,1,0,2,0,0,0,9,0],[0,0,0,0,1,0,2,0,0,0,9,0]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[7,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,4,3,3,0,0,0,0,0,0,0],[0,0,4,0,3,0,8,0,0,0,0,0],[0,0,4,0,3,0,8,0,0,0,0,0],[0,0,6,5,5,5,5,5,0,0,0,0],[0,0,6,0,0,0,0,5,0,0,0,0],[0,0,3,1,1,1,0,2,2,2,9,0],[0,0,3,0,0,1,0,2,0,0,9,0]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[2,1,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,4,3,3,0,0,0,0,0,0,0],[0,0,4,0,3,0,0,0,0,0,0,0],[0,0,6,5,5,5,5,5,0,0,0,0],[0,3,6,0,0,0,0,5,0,0,0,0],[0,3,1,1,1,0,0,5,0,0,0,0],[0,3,0,0,1,0,0,0,0,0,9,0]],"think_tokens":49175,"end_reason":"forced_close","correct":0}]},"run4":{"136b0064":[{"grid":[[0,0,5,0,0,0,0],[0,0,1,1,1,0,0],[0,0,0,2,2,0,0],[0,0,0,6,0,0,0],[0,0,0,6,0,0,0],[3,3,3,3,0,0,0],[6,0,0,0,0,0,0],[6,0,0,0,0,0,0],[1,1,1,0,0,0,0],[0,0,1,1,1,0,0],[0,0,0,0,6,0,0],[0,0,0,0,6,0,0],[0,3,3,3,3,0,0],[2,2,0,0,0,0,0],[0,0,0,0,0,0,0],[0,0,0,0,0,0,0],[0,0,0,0,0,0,0],[0,0,0,0,0,0,0],[0,0,0,0,0,0,0]],"think_tokens":26523,"end_reason":"natural","correct":1}],"20a9e565":[{"grid":[[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"21897d95":[{"grid":[[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9],[9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9,9]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,3,3,3,3,3,3,3,3,3,3,3,3,6,6,6,6,6,6],[2,2,2,2,2,2,4,4,4,4,4,4,4,4,4,4,4,4,6,6,6,6,6,6],[2,2,2,2,2,2,4,4,4,4,4,4,4,4,4,4,4,4,6,6,6,6,6,6],[2,2,2,2,2,2,4,4,4,4,4,4,4,4,4,4,4,4,6,6,6,6,6,6],[2,2,2,2,2,2,4,4,4,4,4,4,4,4,4,4,4,4,6,6,6,6,6,6],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"247ef758":[{"grid":null,"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[0,0,0,0,0,2,9,1,7,1,7,3,1,4,6,1,1,1],[0,0,0,0,0,2,1,0,7,0,7,0,0,0,0,0,0,1],[0,0,0,0,0,2,7,7,7,7,7,7,0,0,0,0,0,7],[0,0,0,0,0,2,1,0,7,0,7,0,0,0,0,0,0,1],[8,8,8,0,0,2,3,0,0,0,0,0,0,4,4,0,0,3],[0,0,0,0,0,2,4,0,0,0,0,0,0,4,0,0,0,4],[0,0,0,0,0,2,1,0,0,0,0,0,4,4,0,0,0,1],[0,0,0,0,0,2,1,0,0,0,0,0,0,6,0,0,0,1],[0,0,0,0,0,2,6,0,0,0,0,0,0,0,6,0,0,6],[0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,6,0,1],[0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,6,1],[0,0,0,0,0,2,1,1,7,1,7,3,1,4,6,1,1,1]],"think_tokens":41799,"end_reason":"natural","correct":0}],"2b83f449":[{"grid":[[8,8,8,8,8,8,8,8,8,8,6,8,8,8,8,8,8],[0,0,0,0,0,0,0,0,0,8,6,8,0,0,0,0,0],[3,8,8,8,8,8,8,8,8,8,6,8,8,6,8,8,8],[0,0,0,0,0,0,0,0,0,0,0,0,8,6,8,0,0],[3,8,8,8,8,8,8,0,3,8,8,8,8,6,8,3,3],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[8,8,8,8,8,8,8,6,8,8,8,8,8,8,8,8,8],[0,0,0,0,0,0,8,6,8,0,0,0,0,0,0,0,0],[8,8,8,8,6,8,8,6,8,8,8,8,8,8,8,8,3],[0,0,0,8,6,8,0,0,0,0,0,0,0,0,0,0,0],[3,3,8,8,6,8,8,8,8,3,0,8,8,8,8,8,3],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[8,8,8,8,8,8,8,8,8,8,6,8,8,8,8,8,8],[0,0,0,0,0,0,0,0,0,8,6,8,0,0,0,0,0],[8,8,8,8,8,8,8,6,8,8,6,8,8,8,8,8,3],[0,0,0,0,0,0,8,6,8,0,0,0,0,0,0,0,0],[8,8,8,6,8,8,8,6,8,8,8,8,6,8,8,8,8],[0,0,8,6,8,0,0,0,0,0,0,8,6,8,0,0,0],[3,3,8,6,8,8,8,0,3,8,8,8,6,8,8,3,3]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"4c3d4a41":[{"grid":[[0,0,0,0,0,0,0,0,0,5,5,5,5,5,5,5,5,5,5,5],[0,0,0,0,0,0,0,0,0,5,0,5,0,6,0,3,0,8,0,5],[0,0,0,0,0,0,0,0,0,5,0,9,0,5,0,5,0,8,0,5],[0,0,0,0,0,0,0,0,0,5,0,5,0,5,0,5,0,8,0,5],[0,0,0,0,0,0,0,0,0,0,0,5,0,5,0,5,0,5,0,5],[0,0,0,0,0,0,0,0,0,0,0,5,5,5,5,5,5,5,0,5],[0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,5],[0,0,0,0,0,0,0,0,0,5,5,5,5,5,5,5,5,5,5,5]],"think_tokens":34958,"end_reason":"natural","correct":1},{"grid":[[0,0,0,0,0,0,0,0,0,5,5,5,5,5,5,5,5,5,5,5],[0,0,0,0,0,0,0,0,0,5,0,3,0,2,0,4,0,7,0,5],[0,0,0,0,0,0,0,0,0,5,0,3,0,2,0,4,0,7,0,5],[0,0,0,0,0,0,0,0,0,5,0,3,0,2,0,4,0,7,0,5],[0,0,0,0,0,0,0,0,0,0,0,5,0,5,0,5,0,5,0,5],[0,0,0,0,0,0,0,0,0,0,0,5,5,5,5,5,5,5,0,5],[0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,5],[0,0,0,0,0,0,0,0,0,5,5,5,5,5,5,5,5,5,5,5]],"think_tokens":29173,"end_reason":"natural","correct":0}],"4c7dc4dd":[{"grid":[[2,2,2,2,2],[2,2,0,2,2],[2,0,0,0,2],[2,2,0,2,2],[2,2,2,2,2]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[0,2,0,0,2,0],[2,0,0,0,0,2],[0,0,0,0,0,0],[0,0,0,0,0,0],[2,0,0,0,0,2],[0,2,0,0,2,0]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"6e4f6532":[{"grid":null,"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3],[3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,1,8,8,8,0,0],[1,1,0,0,0,0,0,0,0,0,0,8,8,8,0,0],[1,1,0,0,0,0,0,0,0,0,0,8,8,8,0,0],[1,1,0,0,0,0,0,0,0,0,0,8,9,8,0,0],[1,1,0,0,0,0,0,0,0,0,0,8,8,8,0,0],[1,1,0,0,0,0,0,0,0,0,0,8,8,8,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,7,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,7,7],[1,1,7,7,7,7,7,7,7,7,7,7,7,7,7,7],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,8,8,8,0,0,0,0,0,0,0,0,0,0],[1,1,0,9,9,8,0,0,0,0,0,0,0,0,0,0],[1,1,0,9,9,8,8,0,0,0,0,0,0,0,0,0],[1,1,0,1,8,8,8,0,0,0,0,0,0,0,0,0],[1,1,0,0,2,0,2,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[1,1,0,0,7,7,7,0,0,0,0,0,0,0,0,0],[1,1,0,0,8,8,8,0,0,0,0,0,0,0,0,0],[1,1,0,0,8,9,8,0,0,0,0,0,0,0,0,0],[1,1,0,0,8,9,8,0,0,0,0,0,0,0,0,0],[1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2],[2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"88bcf3b4":[{"grid":[[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,8,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,8,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,8,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,8,5,5,5,5,5,5,5,9,9,5,5,5,5,5,5,5,5,5,5],[5,5,5,8,5,5,5,5,5,5,9,9,9,5,5,5,5,5,5,5,5,5],[5,5,5,5,8,5,5,5,5,5,9,9,9,9,5,5,5,5,5,5,5,5],[5,5,5,5,5,8,8,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[3,3,3,3,3,3,3,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,6,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,1,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,4,4,4,5,5,1,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,4,5,5,5,5,5,1,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,1,5,5,5,5,5,5,5,5,5,5,5,5,5]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[8,8,8,8,8,8,8,8,8,8,8,8,8,3,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,3,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,3,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,3,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,3,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,3,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,9,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,9,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,9,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,2,2,2,2,2,2],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,5,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,6,6,8,8,8,8,5,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,6,6,8,8,8,5,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,5,5,8,8,8],[8,8,8,8,5,5,5,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,5,5,8,8],[8,8,8,8,5,5,5,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,4,4,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,4,4,4,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,4,8,8,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8],[8,8,8,8,8,8,1,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8,8]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"898e7135":[{"grid":[[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4],[4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4,4]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"8b7bacbf":[{"grid":[[6,1,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,3,8],[1,1,1,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,3,3],[5,5,5,1,9,9,1,1,1,1,1,1,1,9,9,9,9,9,9,9,5,5,5,5,5,5,3,3,5,5],[5,5,5,9,6,6,9,5,5,5,5,5,9,8,8,8,8,8,8,8,9,5,5,5,5,5,3,5,5,5],[5,5,5,9,6,6,9,5,5,5,5,5,9,8,8,8,8,8,8,8,9,5,5,5,5,3,3,5,5,5],[5,5,5,5,9,9,5,5,5,5,5,5,5,9,9,9,9,9,9,9,1,1,1,1,5,3,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,1,3,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,3,1,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,3,5,5,1,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,9,9,3,5,5,1,5,5],[5,5,5,5,9,9,9,9,9,9,9,5,5,5,5,5,9,3,3,5,5,9,5,5,9,5,5,1,5,5],[5,5,5,9,6,6,6,6,6,6,6,9,5,5,5,9,9,9,3,3,3,9,5,5,9,5,5,1,5,5],[5,5,5,9,6,6,6,6,6,6,6,9,5,5,5,9,5,9,5,5,3,3,9,9,5,5,5,1,1,5],[5,5,3,3,9,9,9,9,9,9,9,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,5,5,1,5],[5,0,3,5,5,5,5,5,5,5,5,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,5,5,1,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,5,1,1,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,5,1,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,5,1,5,5],[5,5,0,5,5,5,5,5,5,5,5,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,1,1,5,5],[5,5,3,5,5,5,5,5,5,5,5,5,5,5,5,9,5,9,5,5,5,5,5,5,5,5,1,5,5,5],[5,5,3,5,5,5,5,5,5,5,5,5,5,3,3,3,9,5,5,5,5,5,5,5,5,1,1,5,5,5],[5,5,3,3,5,5,5,5,5,5,3,3,3,3,5,5,5,5,5,5,9,9,9,1,1,1,5,5,5,5],[5,5,5,3,9,9,3,3,3,3,3,5,5,5,5,5,5,5,5,9,5,5,9,1,1,5,5,5,5,5],[5,5,5,9,6,6,9,5,5,5,5,5,5,5,5,5,5,5,9,5,5,5,9,5,5,5,5,5,5,5],[5,5,5,9,6,6,9,5,5,5,5,5,5,5,5,5,5,9,5,5,5,9,5,5,5,5,5,5,5,5],[5,5,5,5,9,9,5,5,5,5,5,5,5,5,5,5,9,5,5,5,9,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,9,5,5,5,9,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,9,9,9,9,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,8,8,0],[0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,1,1,0,0,0,0,0,8,4,4,8],[0,0,0,0,2,2,1,1,0,0,0,0,0,0,0,0,1,1,8,8,0,0,8,4,4,8],[0,0,0,2,0,0,2,0,0,0,0,0,0,0,0,0,0,8,4,4,8,0,0,8,8,0],[0,0,0,2,0,0,2,0,0,0,0,0,0,0,0,0,0,8,4,4,8,0,0,0,0,0],[0,0,0,1,2,2,1,0,0,0,0,0,0,0,0,0,0,0,8,8,1,0,0,0,0,0],[0,0,1,1,0,0,1,1,6,6,6,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0],[0,0,1,0,0,0,0,6,4,4,0,6,0,0,0,8,8,0,0,0,0,0,0,0,1,0],[0,0,1,0,0,0,0,6,4,4,0,6,0,0,8,4,4,8,0,0,0,5,5,5,1,0],[0,0,1,0,0,0,0,6,4,4,0,6,0,0,8,4,4,8,0,0,5,4,4,4,5,0],[0,0,1,0,0,0,0,0,6,6,6,0,0,9,9,8,8,9,0,0,5,4,4,4,5,0],[0,0,1,0,0,0,0,0,0,0,0,0,9,0,0,0,0,9,9,0,0,5,5,5,0,0],[0,0,1,0,0,0,0,0,8,8,9,9,0,0,0,0,0,0,9,0,0,0,0,0,0,0],[0,0,1,0,0,0,0,8,4,4,8,0,0,0,0,0,0,0,9,0,0,0,0,0,0,0],[0,0,1,0,0,0,0,8,4,4,8,0,0,0,0,0,0,0,9,0,0,0,0,0,0,0],[0,0,1,0,0,0,0,0,8,8,0,0,0,0,0,0,0,0,9,0,0,0,8,8,0,0],[0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,8,8,9,0,0,8,4,4,8,0],[0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,8,4,4,8,0,0,8,4,4,8,0],[0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,8,4,4,8,0,0,9,8,8,0,0],[0,1,0,0,0,0,0,9,9,9,9,0,0,0,0,9,8,8,9,9,9,0,0,0,0,0],[1,1,0,0,9,9,9,0,0,0,9,9,9,9,9,9,0,0,0,0,0,0,0,0,0,0],[4,1,0,0,9,3,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]],"think_tokens":49175,"end_reason":"forced_close","correct":0}],"b6f77b65":[{"grid":[[3,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,4,3,3,0,0,0,0,0,0],[0,0,0,0,3,0,8,7,7,7,0,0],[0,0,0,0,3,0,8,0,0,7,0,0],[0,0,0,0,3,0,8,0,0,7,0,0],[0,0,6,5,5,5,5,5,0,7,0,0],[0,0,6,0,0,0,0,5,0,7,0,0]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[7,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,4,3,3,0,0,0,0,0,0,0],[0,0,4,0,3,0,8,0,0,0,0,0],[0,0,4,0,3,0,8,0,0,0,0,0],[0,0,6,5,5,5,5,5,0,0,0,0],[0,0,6,0,0,0,0,5,0,0,0,0],[0,3,1,1,1,0,2,2,2,2,9,0]],"think_tokens":49175,"end_reason":"forced_close","correct":0},{"grid":[[2,1,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,0,0,0,0,0,0,0,0,0,0],[0,0,4,3,3,0,0,0,0,0,0,0],[0,0,4,0,3,0,8,0,0,0,0,0],[0,0,4,0,3,0,8,7,7,7,0,0],[0,0,6,5,5,5,5,5,0,7,0,0],[0,0,6,0,0,0,0,5,0,7,0,0],[0,3,1,1,1,0,0,0,0,7,9,0],[0,3,0,0,1,0,0,0,0,0,9,0],[0,3,0,0,1,0,0,0,0,0,9,0]],"think_tokens":49175,"end_reason":"forced_close","correct":0}]}}

In [ ]:
%%writefile /kaggle/working/agent/run_eval.py
"""Commit-run evaluation: the REPL agent on a fixed set of evaluation tasks, scored against the public solutions.

    python run_eval.py '<json config>'   (normally called by submit_main once the server is up)

Writes /kaggle/working/agent_eval/attempts.jsonl (one line per finished attempt, with transcript) and
eval_report.json (score, per task, tokens, turns, statuses). Never touches /kaggle/working/submission.json.
"""
import glob
import json
import os
import re
import sys
import threading
import time
from collections import Counter
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from client import ChatClient                      # noqa: E402
from repl_agent import solve_repl, AGENT_OPTS, breaker, ABORT   # noqa: E402
from agent import to_submission, score             # noqa: E402
from rescue import run_rescue, hybrid_submission, direct_submission, hits, load_direct_ref   # noqa: E402  (v1.6/1.8)


def find(name, root="/kaggle/input"):
    found = glob.glob(f"{root}/**/{name}", recursive=True)
    if not found:
        raise FileNotFoundError(name)
    return found[0]


def main(cfg):
    t0 = time.time()
    root = cfg.get("data_dir", "/kaggle/input")
    ch = json.load(open(find("arc-agi_evaluation_challenges.json", root)))
    sol = json.load(open(find("arc-agi_evaluation_solutions.json", root)))
    ids = [t for t in cfg["eval_task_ids"] if t in ch]
    tasks = {t: ch[t] for t in ids}
    out = Path(cfg.get("eval_out", "/kaggle/working/agent_eval"))
    out.mkdir(parents=True, exist_ok=True)
    client = ChatClient(cfg["base_url"], cfg["model"], max_tokens=cfg.get("max_tokens", 16384),
                        temperature=cfg.get("temperature", 0.6), top_p=cfg.get("top_p", 0.95),
                        top_k=cfg.get("top_k", 20), thinking=cfg.get("thinking", True),
                        reasoning_effort=cfg.get("reasoning_effort"),
                        thinking_token_budget=cfg.get("thinking_token_budget"),
                        template_kwargs=cfg.get("template_kwargs"))
    lock = threading.Lock()
    jsonl = open(out / "attempts.jsonl", "w", encoding="utf-8")     # v1.1: fresh log per run
    progress = {"attempts_done": 0, "verified_tasks": set(), "gen_tokens": 0, "statuses": Counter()}

    def on_done(rec):
        breaker(rec)                                   # v1.6: stop early if the first attempts all die on HTTP errors
        with lock:
            jsonl.write(json.dumps(rec, default=str) + "\n")
            jsonl.flush()
            # v1.1: a progress snapshot after every attempt survives a deadline cut or a killed process
            progress["attempts_done"] += 1
            progress["gen_tokens"] += rec.get("gen_tokens", 0)
            progress["statuses"][rec.get("status")] += 1
            if rec.get("status") == "verified":
                progress["verified_tasks"].add(rec["task_id"])
            (out / "progress.json.tmp").write_text(json.dumps(
                {"attempts_done": progress["attempts_done"], "verified_tasks": sorted(progress["verified_tasks"]),
                 "gen_tokens": progress["gen_tokens"], "statuses": dict(progress["statuses"]),
                 "elapsed_s": round(time.time() - t0)}, indent=1))
            os.replace(out / "progress.json.tmp", out / "progress.json")     # v1.2: atomic

    # v1.6: the agent gets agent_frac of the time; the rest is for DIRECT rescue answers.
    # v1.8: eval_direct "ref" (default) merges with the LB-18.75 pipeline's recorded EVAL12 answers instead of
    # re-generating them (~1.1M tokens, ~70 GPU-min) -> the agent keeps the whole eval time; "live" re-runs DIRECT
    direct_mode = cfg.get("eval_direct", "ref") if cfg.get("rescue") else "off"
    agent_deadline = (t0 + (cfg["deadline_epoch"] - t0) * float(cfg.get("agent_frac", 0.6))
                      if direct_mode == "live" else cfg["deadline_epoch"])
    print(f"eval: {len(tasks)} tasks, agent deadline in {(agent_deadline - time.time()) / 60:.0f} min, "
          f"final deadline in {(cfg['deadline_epoch'] - time.time()) / 60:.0f} min", flush=True)
    answers, report = solve_repl(
        tasks, client, attempts=cfg.get("attempts", 2), concurrency=cfg.get("concurrency", 16),
        deadline=agent_deadline, stop_after=1, max_turns=cfg.get("max_turns", 40),
        max_gen_tokens=cfg.get("eval_cap", 60000), context_budget=cfg.get("context_budget", 28000),
        nudge_every=cfg.get("nudge_every", 0), turn_max_tokens=cfg.get("turn_max_tokens"),
        in_python=cfg.get("in_python", False), log=lambda m: print(m, flush=True), on_done=on_done,
        **{k: cfg[k] for k in AGENT_OPTS if k in cfg and k not in ("turn_max_tokens", "in_python")})
    jsonl.close()
    # v1.6 HYBRID: DIRECT answers for tasks the agent did not verify (rescue_all: every task, to measure overlap)
    atts0 = report["attempts"]
    verified0 = {a["task_id"] for a in atts0 if a["status"] == "verified"}
    rescue_recs, rescue_client = {}, None
    rescue_t0 = rescue_t1 = time.time()
    ref_path = Path(__file__).with_name("direct_ref_eval12.json")
    ref_run4 = {}
    if direct_mode == "ref":
        ref = load_direct_ref(ref_path, cfg.get("eval_direct_run", "run7"))
        rescue_recs = {t: ref[t] for t in ids if t in ref and (cfg.get("rescue_all") or t not in verified0)}
        ref_run4 = {t: r for t, r in load_direct_ref(ref_path, "run4").items() if t in ids}
        print(f"DIRECT from the LB-18.75 run records ({cfg.get('eval_direct_run', 'run7')}): {len(rescue_recs)} tasks",
              flush=True)
    elif direct_mode == "live":
        ids_r = ids if cfg.get("rescue_all") else [t for t in ids if t not in verified0]
        print(f"rescue (direct answers): {len(ids_r)} tasks, {(cfg['deadline_epoch'] - time.time()) / 60:.0f} min left",
              flush=True)
        rjsonl = open(out / "rescue_attempts.jsonl", "w", encoding="utf-8")
        done_r = {}

        def on_rescue(tid, rec):                       # v1.7: durable record + deployed snapshot after every answer
            rjsonl.write(json.dumps(dict(rec, task_id=tid), default=str) + "\n")
            rjsonl.flush()
            done_r[tid] = rec
            snap = hybrid_submission(answers, {t: r for t, r in done_r.items() if t not in verified0}, tasks)
            (out / "eval_submission.partial.json.tmp").write_text(json.dumps(snap))
            os.replace(out / "eval_submission.partial.json.tmp", out / "eval_submission.partial.json")

        rescue_t0 = time.time()
        rescue_recs, rescue_client = run_rescue(ids_r, tasks, cfg, cfg["deadline_epoch"],
                                                log=lambda m: print(m, flush=True), on_result=on_rescue)
        rescue_t1 = time.time()
        rjsonl.close()
    sol_ids = {t: sol[t] for t in ids}
    agent_sub = to_submission(answers, tasks)
    direct_sub = direct_submission(rescue_recs, tasks)
    # v1.7 (GPT final audit): the DEPLOYED policy rescues only tasks the agent did not verify -- that is the score that
    # predicts the hidden run; the all-tasks-rescued merge is a counterfactual reported beside it
    deployed_rescue = {t: r for t, r in rescue_recs.items() if t not in verified0}
    sub = hybrid_submission(answers, deployed_rescue, tasks)
    all_sub = hybrid_submission(answers, rescue_recs, tasks)
    h_agent, h_direct, h_hybrid, h_all = (hits(x, sol_ids, ids) for x in (agent_sub, direct_sub, sub, all_sub))
    rescued_q = [(t, j) for t in rescue_recs for j in range(len(sol[t]))]
    r_status = Counter(str(u.get("status")).split(":")[0] for r in rescue_recs.values()
                       for u in (r.get("units") or {}).values())             # v1.8: per test input
    complementarity = {
        "queries": len(h_agent),
        "agent_correct": sum(h_agent.values()), "direct_correct": sum(h_direct.values()),
        "hybrid_correct": sum(h_hybrid.values()), "hybrid_all_rescued_correct": sum(h_all.values()),
        "agent_only": sum(1 for k in h_agent if h_agent[k] and not h_direct[k]),
        "direct_only": sum(1 for k in h_agent if h_direct[k] and not h_agent[k]),
        "both": sum(1 for k in h_agent if h_agent[k] and h_direct[k]),
        "oracle_union_hits": sum(1 for k in h_agent if h_agent[k] or h_direct[k]),   # retrospective bound, not a policy
        "hybrid_lost_vs_agent": sum(1 for k in h_agent if h_agent[k] and not h_hybrid[k]),
        "hybrid_lost_vs_direct": sum(1 for k in h_agent if h_direct[k] and not h_hybrid[k]),
        "all_rescued_lost_vs_agent": sum(1 for k in h_agent if h_agent[k] and not h_all[k]),
        "all_rescued_lost_vs_direct": sum(1 for k in h_agent if h_direct[k] and not h_all[k]),
        "direct_correct_on_unverified_queries": sum(h_direct[k] for k in h_direct if k[0] not in verified0),
        "direct_correct_on_rescued_queries": sum(h_direct[k] for k in rescued_q),
        "rescued_queries": len(rescued_q),
        "rescue_statuses": dict(r_status),
        "rescue_gen_tokens": sum(r.get("gen_tokens", 0) for r in rescue_recs.values()),
        "rescue_budget_dropped": bool(getattr(rescue_client, "budget_dropped", False)),
        "rescue_smoke_ok": cfg.get("rescue_smoke_ok"),
        "direct_source": {"ref": "LB-18.75 pipeline records (" + str(cfg.get("eval_direct_run", "run7")) + ")",
                          "live": "generated in this run", "off": "none"}[direct_mode],
        # v1.7: the three-way comparison is only complete if DIRECT really answered (or tried) every task
        "three_way_complete": bool(direct_mode != "off" and cfg.get("rescue_all")
                                   and (direct_mode == "ref" or cfg.get("rescue_smoke_ok", True))
                                   and len(rescue_recs) == len(ids)
                                   and not any(s in r_status for s in ("skipped", "crashed", "error"))),
    }
    if ref_run4:                                       # v1.8: the same merge against the other recorded DIRECT run
        h4 = hits(direct_submission(ref_run4, tasks), sol_ids, ids)
        h4h = hits(hybrid_submission(answers, {t: r for t, r in ref_run4.items() if t not in verified0}, tasks),
                   sol_ids, ids)
        complementarity["reference_run4"] = {"direct_correct": sum(h4.values()), "hybrid_correct": sum(h4h.values()),
                                             "agent_only_vs_run4": sum(1 for k in h_agent if h_agent[k] and not h4[k])}
    total, pct, per = score(sub, sol_ids)
    _, _, agent_per = score(agent_sub, sol_ids)
    q_tot = len(h_hybrid)
    q_hit = sum(h_hybrid.values())
    atts = report["attempts"]
    agent_gen = sum(a["gen_tokens"] for a in atts)
    agent_wall = float(report.get("wall_s") or (rescue_t0 - t0))
    rescue_gen = complementarity["rescue_gen_tokens"]
    rescue_wall = max(0.0, rescue_t1 - rescue_t0) if direct_mode == "live" else 0.0
    if direct_mode != "live":
        rescue_gen = 0                                 # reference tokens were spent in runs 4 / 7, not here
    wall = time.time() - t0
    official_pct = 100.0 * q_hit / q_tot if q_tot else 0.0
    verified_tasks = sorted({a["task_id"] for a in atts if a["status"] == "verified"})
    scout_cap = (cfg.get("rerun_attempt_caps") or cfg.get("attempt_caps") or [0])[0]
    # v1.2: generated code runs with the real filesystem, where the evaluation SOLUTIONS file exists. Flag every task
    # whose executed code touched files/paths so a score cannot silently come from reading answers.
    # v1.3: wider net (audit R3 read a file via builtins) -- this is DETECTION for manual review, not a sandbox
    leak_re = re.compile(r"solution|/kaggle/|open\s*\(|listdir|scandir|glob|pathlib|Path\s*\(|walk\s*\(|builtins|"
                         r"__import__|importlib|\bio\b|\bos\b|\bsys\b|subprocess|shutil|socket|eval\s*\(|exec\s*\(|"
                         r"compile\s*\(|getattr\s*\(|globals\s*\(|locals\s*\(|vars\s*\(|ctypes|cffi|mmap", re.I)
    suspicious = sorted({a["task_id"] for a in atts for x in a.get("transcript", [])
                         if isinstance(x, dict) and isinstance(x.get("code"), str) and leak_re.search(x["code"])})
    tdump = {id(a): json.dumps(a.get("transcript", []), default=str) for a in atts}
    summary = {
        # v1.7 (GPT final audit): from the AGENT-only submission -- a DIRECT rescue must not hide a wrong verified program
        "verified_but_wrong_tasks": [t for t in verified_tasks if agent_per.get(t, 0) < 1],   # train-fit != test-correct
        "verified_but_wrong_queries": [f"{t}#{j}" for t in verified_tasks for j in range(len(sol[t]))
                                       if not h_agent[(t, j)]],
        "file_access_suspicious_tasks": suspicious,
        # v1.7: what the guard actually stopped, and children that refused to run because the guard failed
        "isolation_denied_tasks": sorted({a["task_id"] for a in atts if "[sandbox] access denied" in tdump[id(a)]
                                          or "[sandbox] not allowed" in tdump[id(a)]}),
        "isolation_guard_failed_tasks": sorted({a["task_id"] for a in atts if "[sandbox] guard failed" in tdump[id(a)]}),
        "thinking_budget_dropped_at_runtime": bool(getattr(client, "budget_dropped", False)),   # v1.3
        # v1.1: Kaggle's metric = correct test outputs / all test outputs (micro); the task average is kept beside it
        # v1.7: official = the DEPLOYED hybrid (rescue only where the agent did not verify)
        "official_score_pct": round(official_pct, 2), "task_macro_score_pct": round(pct, 2),
        "hybrid_all_rescued_pct": round(100.0 * complementarity["hybrid_all_rescued_correct"] / max(q_tot, 1), 2),
        "agent_only_official_pct": round(100.0 * complementarity["agent_correct"] / max(q_tot, 1), 2),
        "direct_only_official_pct": round(100.0 * complementarity["direct_correct"] / max(q_tot, 1), 2),
        "complementarity": complementarity,
        "circuit_breaker": dict(ABORT),
        "agent_deadline_epoch": agent_deadline,
        "score_tasks": round(total, 3), "score_pct": round(pct, 2), "n_tasks": len(ids),
        "query_hits": q_hit, "queries": q_tot, "per_task": per, "per_task_agent_only": agent_per,
        "tasks_with_verified_program": len(verified_tasks),
        # v1.7: how many verifications the hidden run's smaller scout cap would have kept (counterfactual)
        "verified_first_attempt_within_rerun_scout_cap": {"cap": scout_cap, "tasks": sorted(
            {a["task_id"] for a in atts if a["status"] == "verified" and a["idx"] == 0
             and a["gen_tokens"] <= scout_cap})},
        "status_counts": dict(Counter(a["status"] for a in atts)),
        "notes": dict(Counter(a["note"] for a in atts if a["note"])),
        # v1.7 (GPT final audit): throughput per stage (the old figure divided agent tokens by agent + rescue time)
        "agent_generated_tokens": agent_gen, "agent_wall_s": round(agent_wall, 1),
        "agent_gen_tok_per_s": round(agent_gen / max(agent_wall, 1e-6), 1),
        "rescue_generated_tokens": rescue_gen, "rescue_wall_s": round(rescue_wall, 1),
        "rescue_gen_tok_per_s": round(rescue_gen / max(rescue_wall, 1e-6), 1),
        "combined_gen_tok_per_s": round((agent_gen + rescue_gen) / max(wall, 1e-6), 1),
        "gen_tokens_total": agent_gen + rescue_gen, "gen_tokens_mean_per_attempt": round(agent_gen / max(len(atts), 1)),
        "turns_mean": round(sum(a["turns"] for a in atts) / max(len(atts), 1), 1),
        "cuts_total": sum(a["cuts"] for a in atts), "submits_total": sum(a["submits"] for a in atts),
        "python_calls_total": sum(a["python_calls"] for a in atts),
        "wall_s": round(wall),
        "config": {k: v for k, v in cfg.items() if k not in ("base_url",)},
        "per_attempt": [{k: a[k] for k in ("task_id", "idx", "status", "note", "turns", "python_calls", "submits",
                                           "cuts", "forced", "gen_tokens", "prompt_tokens", "wall", "best_ok",
                                           "n_train", "effort_steps", "evictions")} for a in atts]}
    (out / "eval_report.json").write_text(json.dumps(summary, indent=1, default=str))
    (out / "eval_submission.json").write_text(json.dumps(sub))                     # the deployed policy
    (out / "eval_all_rescued_submission.json").write_text(json.dumps(all_sub))     # counterfactual
    print(f"\nscores (official = deployed hybrid): hybrid {official_pct:.1f}% | all-rescued "
          f"{summary['hybrid_all_rescued_pct']}% | agent {summary['agent_only_official_pct']}% | direct "
          f"{summary['direct_only_official_pct']}% | {complementarity}", flush=True)
    print(f"\nEVAL OFFICIAL {official_pct:.1f}% = queries {q_hit}/{q_tot} | task average {total:.2f} / {len(ids)} "
          f"({pct:.1f}%) | verified tasks {summary['tasks_with_verified_program']} "
          f"(wrong on test: {summary['verified_but_wrong_tasks']}) | agent {agent_gen} tok "
          f"({summary['agent_gen_tok_per_s']} tok/s) | rescue {rescue_gen} tok ({summary['rescue_gen_tok_per_s']} tok/s) "
          f"| wall {wall / 60:.0f} min | {summary['status_counts']}", flush=True)
    print("per task:", {t: round(v, 2) for t, v in per.items()}, flush=True)

if __name__ == "__main__":
    main(json.loads(sys.argv[1]))


In [ ]:
# ==================================================================== launch: evaluation (commit run) or hidden test (rerun)
import json, os, subprocess, sys, time
from pathlib import Path
START = float(Path("/kaggle/working/start_epoch.txt").read_text())
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
EVAL12 = ["136b0064", "20a9e565", "21897d95", "247ef758", "2b83f449", "4c3d4a41", "4c7dc4dd", "6e4f6532", "88bcf3b4", "898e7135", "8b7bacbf", "b6f77b65"]
SERVER = json.loads(r"""{"model_glob": "/kaggle/input/**/qwen3-8-27b-fp8-repacked-v1/**/config.json", "overlay": "/kaggle/input/**/taaf-duck-qwen38-serving-v1/serving_wheels/*.whl", "parsers": ["--tool-call-parser", "qwen3_coder", "--reasoning-parser", "qwen3"], "server_ladder": [["--reasoning-config", "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}", "--kv-cache-dtype", "fp8", "--language-model-only", "--speculative-config", "{\"method\": \"qwen3_5_mtp\", \"num_speculative_tokens\": 2}"], ["--reasoning-config", "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}", "--kv-cache-dtype", "fp8", "--speculative-config", "{\"method\": \"qwen3_5_mtp\", \"num_speculative_tokens\": 2}"], ["--reasoning-config", "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}", "--kv-cache-dtype", "fp8", "--speculative-config", "{\"method\": \"mtp\", \"num_speculative_tokens\": 2}"], ["--reasoning-config", "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}", "--kv-cache-dtype", "fp8"], ["--reasoning-config", "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}"], []], "max_model_len": 81920, "max_num_seqs": 16}""")
AGENT = json.loads(r"""{"mode": "repl", "attempts": 2, "concurrency": 16, "temperature": 0.6, "top_p": 0.95, "top_k": 20, "template_kwargs": {"reasoning_effort": "xhigh"}, "thinking_token_budget": 6144, "max_tokens": 8192, "turn_max_tokens": 8192, "max_turns": 30, "context_budget": 28000, "keep_recent": 6, "trim_drain": 6000, "nudge_every": 4, "in_python": true, "force_submit_after": 5, "effort_ladder": ["medium", "low"], "retain_functions": true, "degenerate_guard": true, "timeout_as_cut": true, "snippet_timeout": 30.0, "tok_per_s": 300, "min_cap": 15000, "max_cap": 60000, "eval_cap": 60000, "cap_factor": 1.0, "max_num_batched_tokens": 4096, "attempt_caps": [24000, 36000], "rerun_attempt_caps": [16000, 32000], "e2e_max_s": 600, "rescue": true, "agent_frac": 0.5, "rescue_effort": "xhigh", "rescue_think": 32768, "rescue_answer": 4096, "rescue_commit_tokens": 4096, "rescue_temperature": 0.6, "rescue_min_start_s": 300}""")
cfg = dict(SERVER, **AGENT)
if RERUN:
    # hidden test: the agent stops at 11 h 35 min; the process is cut at 11 h 50 min (10 min before the 12 h limit)
    cfg["deadline_epoch"] = START + 12 * 3600 - 25 * 60
    cfg["min_agent_s"] = 3600            # the server ladder never starts a rung with < 1 h left for the agent
else:
    EVAL_HOURS = float(os.environ.get("Q27A_EVAL_HOURS", 2.0))
    _ids = os.environ.get("Q27A_EVAL_TASKS", "eval12").strip()
    cfg.update(eval=True, deadline_epoch=START + EVAL_HOURS * 3600, min_agent_s=1800,
               eval_task_ids=EVAL12 if _ids == "eval12" else [t.strip() for t in _ids.split(",") if t.strip()])
    # v1.6: the commit run rescues EVERY task with a direct answer, so the report shows agent-only / direct-only /
    # hybrid scores and their overlap; the agent gets 60 % of the eval time
    cfg.update(rescue_all=True, agent_frac=float(os.environ.get("Q27A_AGENT_FRAC", 0.6)))
print("mode:", "COMPETITION RERUN (hidden test)" if RERUN else f"EVALUATION ({len(cfg['eval_task_ids'])} tasks)",
      "| agent deadline in", round((cfg["deadline_epoch"] - time.time()) / 60), "min", flush=True)
print(json.dumps({k: v for k, v in cfg.items() if k not in ("server_ladder",)}, indent=1), flush=True)


def kill_leftover_server():
    """v1.1: subprocess.run's timeout SIGKILLs submit_main, whose finally-cleanup then never runs, and the vLLM
    server lives in its own session. Kill that recorded process group (only if it is a vLLM server or already gone)."""
    import signal
    pid_file = Path("/kaggle/working/vllm_server.pid")
    if not pid_file.exists():
        return
    try:
        pid = int(pid_file.read_text().strip())
        cmd = Path(f"/proc/{pid}/cmdline")
        if (not cmd.exists()) or ("vllm" in cmd.read_bytes().decode(errors="replace")):
            os.killpg(pid, signal.SIGKILL)
            print("killed leftover vLLM server process group", pid, flush=True)
    except ProcessLookupError:
        pass
    except Exception as e:
        print("server cleanup:", repr(e), flush=True)


try:
    rc = subprocess.run([sys.executable, "/kaggle/working/agent/submit_main.py", json.dumps(cfg)],
                        timeout=max(60, cfg["deadline_epoch"] + 15 * 60 - time.time())).returncode
    print("submit_main exit code", rc)
except Exception as e:
    print("submit_main did not finish cleanly:", repr(e))
finally:
    kill_leftover_server()
for _p in ("/kaggle/working/agent_server_report.json", "/kaggle/working/agent_eval/eval_report.json",
           "/kaggle/working/agent_eval/progress.json"):
    try:
        _r = json.loads(Path(_p).read_text())
        if _p.endswith("eval_report.json"):
            _r = {k: _r.get(k) for k in ("official_score_pct", "query_hits", "queries", "task_macro_score_pct",
                                         "hybrid_all_rescued_pct", "agent_only_official_pct",
                                         "direct_only_official_pct", "complementarity", "verified_but_wrong_tasks",
                                         "tasks_with_verified_program", "status_counts", "circuit_breaker",
                                         "isolation_denied_tasks", "isolation_guard_failed_tasks",
                                         "agent_gen_tok_per_s", "rescue_gen_tok_per_s", "gen_tokens_total", "wall_s")}
        print(_p, json.dumps(_r, default=str)[:3000], flush=True)
    except Exception as e:
        print("no", _p, repr(e)[:120], flush=True)


In [ ]:
# Last line of defence (v1.2, rewritten): whatever was left behind, submission.json ends up valid -- every hidden
# task, every test input, two grids of plain ints 0-9 with 1..30 rows and columns. A bad ROW is replaced by the
# fallback (test input); a non-dict file, null/str rows, ragged or out-of-range grids and unknown task keys are handled.
import json, glob, os
PATH = "/kaggle/working/submission.json"


def ok_grid(g):
    return (isinstance(g, list) and 1 <= len(g) <= 30
            and all(isinstance(r, list) and 1 <= len(r) <= 30 and len(r) == len(g[0]) for r in g)
            and all(type(v) is int and 0 <= v <= 9 for r in g for v in r))


try:
    ch = json.load(open(glob.glob("/kaggle/input/**/arc-agi_test_challenges.json", recursive=True)[0]))
    try:
        raw = json.load(open(PATH))
    except Exception:
        raw = {}
    if not isinstance(raw, dict):
        raw = {}
    fixed, n_bad, answered = {}, 0, 0
    for t, task in ch.items():
        rows = raw.get(t)
        rows = rows if isinstance(rows, list) and len(rows) == len(task["test"]) else [None] * len(task["test"])
        out_rows = []
        for r, x in zip(rows, task["test"]):
            if isinstance(r, dict) and ok_grid(r.get("attempt_1")) and ok_grid(r.get("attempt_2")):
                out_rows.append({"attempt_1": r["attempt_1"], "attempt_2": r["attempt_2"]})
                answered += int(r["attempt_1"] != x["input"])
            else:
                n_bad += 1
                out_rows.append({"attempt_1": x["input"], "attempt_2": x["input"]})
        fixed[t] = out_rows
    unknown = [t for t in raw if t not in ch]
    with open(PATH + ".tmp", "w") as f:
        json.dump(fixed, f)
    os.replace(PATH + ".tmp", PATH)
    print(f"submission.json valid: {len(fixed)} tasks, {answered} test outputs answered by the agent, "
          f"{n_bad} rows replaced by the fallback, {len(unknown)} unknown task keys dropped")
except Exception as e:
    print("validation error:", repr(e))
